# Citi Bike Demand During G Train Disruptions

## DATA 622 Final Project

This notebook documents the study of Citi Bike demand near G train stations in Greenpoint. It presents methods, data checks, and exploratory findings. This version runs on Databricks and reads its raw data from a Unity Catalog volume that scheduled jobs keep up to date. Disruption effects are described only as screening comparisons from confirmed closure windows and are not treated as causal estimates.

## Contents

1. Problem definition: research question and objectives
2. Data preparation: collection, understanding, and cleaning
   - Data pipeline in Databricks: jobs, cleaning rules, catalog tables and data access
   - 2.1 to 2.4 Sources, collection, quality checks and integration (including 2.2.1 confirmed G closure windows)
   - 2.5 Exposure definition and analysis panel
   - 2.6 Extended datasets for modeling: shutdown table, extended panel, baseline, status proxy and 2024 check
3. Exploration: exploratory data analysis and feature engineering
   - 3.1 to 3.4 Coverage, daily and hourly demand, stations and availability
   - 3.5 Feature engineering
   - 3.6 to 3.9 Exposure, weather, outcome distribution, dataset readiness and summary
4. Model development
5. Evaluation and validation
6. Production design and monitoring
7. Results and discussion
8. Conclusion
9. References and appendices


### Figure and table standard

Each figure will have a descriptive title, labeled axes, units, and a caption that identifies the data and time period. A short interpretation will follow each figure. It will describe the visible pattern, explain why the pattern matters, and state any limits on what the figure can show. A descriptive pattern alone will not be called a causal effect.

# 1. Problem definition: research question and objectives

## 1.1 Background

Citi Bike provides a transportation option when subway service is disrupted. This project focuses on Greenpoint, where the G train is an important subway connection. A service interruption may change the number and direction of trips near affected stops. It may also leave some stations with too few bikes for departing riders or too few open docks for arriving riders. The size, location, and timing of these changes matter to bike operations staff and public transport planners.

The existing project proposal describes this as a study of G train disruptions and Citi Bike demand. This notebook separates the analysis from the proposal and will document the data and evidence used to answer the question.

## 1.2 Research question and hypotheses

**How does a planned or unplanned G train service disruption relate to Citi Bike trip activity and station availability near affected Greenpoint subway stops, compared with expected conditions when service is operating?**

The word *relate* is deliberate. The primary question is whether demand and availability differ during verified G train disruptions, not whether any difference is automatically caused by the disruption. A causal interpretation requires credible comparison periods, correct event dates, and consideration of other causes such as weather, season, holidays, and routine travel patterns.

The working hypotheses are that trip departures and arrivals will change near affected stations during service interruptions, and that the probability of a station reporting no available bikes or no open docks will also change. These are hypotheses to test, not findings. The direction and size of the changes may differ by station, time of day, and day type.

## 1.3 Objectives and success criteria

1. Describe the trip activity and bike availability patterns at Citi Bike stations near affected G train stops.
2. Compare disruption periods with suitable periods when G train service was operating, while accounting for time, day, weather, and station differences where data allow.
3. Identify stations and time periods that may need extra bikes or open docks during a disruption.
4. Prepare a documented analysis dataset and clear definitions that can support later model development.

The first stage is complete when the study population, event definition, data coverage, cleaning decisions, and station by hour measures are documented; the main temporal and spatial patterns are shown in figures; and each figure has an evidence based interpretation. A claim about a disruption effect will require a separate comparison design and supporting event data.

## 1.4 Scope and intended use

The initial area is screened with a rectangular latitude and longitude boundary around Greenpoint. This is a practical way to identify candidate stations, not an official neighborhood boundary or a definition of exposure to a G train closure. The final exposed station group should be based on measured distance to subway stops affected in each verified event. The local trip files cover May to September 2024 and January to September 2026. Historical station status is available from August 21 to October 1, 2026, and only for Greenpoint. The local weather archive covers May to September 2024 and January to September 2026. A live MTA alert snapshot adds 28 G service event periods from October 19 to November 23, 2026, and none overlap the station status period. The MTA subway alert history currently ends on August 11, 2026, before the station status period, so it cannot identify closure hours in the status sample. Hourly ridership at G stations is used to infer them in the inferred windows subsection of Section 2.2. Citi Bike operations staff and public transport planners are the intended users of later estimates.

## 1.5 Population, exposure, and outcomes

The planned unit of analysis is one Citi Bike station in one local clock hour. The initial population is the set of stations inside the Greenpoint screening boundary that can be matched to station information. The exposure is whether a station is near a subway stop during a verified period when the G train is not operating. The exposure cannot be assigned from the current local files alone.

The primary outcomes are the number of departures and arrivals recorded for a station hour. Secondary outcomes describe availability from station status reports: mean reported bikes available, mean reported docks available, and the share of status observations reporting zero bikes or zero docks. A trip count measures completed journeys; a station status report measures inventory at a snapshot. They are complementary outcomes and will not be treated as equivalent measures of demand.

# 2. Data preparation: collection, understanding, and cleaning

This section explains where the data comes from, how it is collected and cleaned, and which tables the analysis uses. The data work runs in **Databricks**, not in this notebook. Three scheduled Databricks jobs download the raw data, clean it and store the results as tables in the Unity Catalog. The notebook then reads those tables. This keeps the 20 GB of raw trip files out of the notebook, makes every run start from the same cleaned data, and lets new data arrive without anyone rerunning a download.

The rest of this introduction describes the pipeline: the jobs (A), the cleaning rules (B), the tables in the catalog (C) and how the notebook reads them (D). Sections 2.1 to 2.6 then document each source and show the checks and derived tables in detail.

### Overview of the data pipeline

```mermaid
flowchart TB
    subgraph SRC["Public data sources"]
        direction LR
        S1["Citi Bike trip archives<br/>monthly ZIP files on Amazon S3"]
        S2["MTA and NY Open Data APIs<br/>alerts, incidents, stations, ridership"]
        S3["Open-Meteo<br/>hourly weather archive"]
        S4["Citi Bike GBFS<br/>live station feeds"]
        S5["Station status and station<br/>information export"]
    end

    subgraph JOBS["Databricks jobs (serverless)"]
        direction LR
        J1["<b>citibike-download</b><br/>Mondays 06:00<br/>downloads and unzips trip months"]
        J2["<b>external-data-download</b><br/>daily 05:00<br/>MTA, weather and GBFS files"]
        J3["<b>citibike-prepare</b><br/>after each weekly download<br/>cleans files, builds tables"]
    end

    subgraph UC["Unity Catalog: workspace.citibike"]
        V[("<b>Volume raw/datasource</b><br/>raw files, unchanged")]
        subgraph TABLES["Delta tables"]
            direction LR
            T1[("<b>Raw tables</b><br/>raw_* (10)<br/>reference files as tables")]
            T2[("<b>Cleaned tables</b> (7)<br/>trips, trips_station_hourly,<br/>station_status_hourly, stations,<br/>weather_hourly, mta_g_stops")]
            T3[("<b>Analysis tables</b> (11)<br/>station hour panels, station sets,<br/>closures, baselines, quality checks")]
        end
    end

    NB["<b>This notebook</b><br/>Sections 2 to 6: exploration,<br/>features, models, evaluation"]

    S1 --> J1
    S2 --> J2
    S3 --> J2
    S4 --> J2
    S5 -.->|uploaded once| V
    J1 -->|trip CSV files| V
    J2 -->|MTA, weather, GBFS files| V
    J1 ==>|then starts| J3
    V --> J3
    J3 --> T1
    J3 --> T2
    J3 --> T3
    T1 -->|read_table| NB
    T2 -->|read_table| NB
    T3 -->|read_table| NB

    classDef source fill:#eef2f7,stroke:#5b6b82,color:#1f2a37
    classDef job fill:#fdebd3,stroke:#c26a00,color:#3d2200
    classDef storage fill:#e3f1e6,stroke:#2f7d43,color:#123d1e
    classDef consumer fill:#e6e9fb,stroke:#3f4fbf,color:#141c4d
    class S1,S2,S3,S4,S5 source
    class J1,J2,J3 job
    class V,T1,T2,T3 storage
    class NB consumer
```

*Figure 0. Data pipeline.* Grey boxes are the public sources, orange boxes the Databricks jobs, green shapes the storage in the Unity Catalog and blue the analysis. Solid arrows are scheduled data flows, the dotted arrow is the one time upload of the station export, and the thick arrow shows that `citibike-download` starts `citibike-prepare` when it finishes. The notebook reads every table with `read_table` (part D below).

The pipeline is defined in the project repository as a Databricks Asset Bundle (`databricks.yml` with one job definition per file in `resources/`). The job notebooks are in `jobs/`, and the code they share with this notebook is in `src/citibike/`. Deploying the bundle creates the jobs in the workspace; the `dev` target keeps the schedules paused for testing and the `prod` target turns them on.

### A. Databricks jobs

All three jobs run on serverless compute (the only option on Databricks Free Edition). Each one takes the catalog, schema and volume as parameters (defaults `workspace`, `citibike`, `raw`), retries automatically on failure, and emails the owner if it fails or runs much longer than usual.

| Job | Notebook | When it runs | What it does | Output |
|---|---|---|---|---|
| `citibike-download` | `jobs/citibike_download.py` | Mondays 06:00 New York time | Downloads each monthly Citi Bike trip archive (May to September 2024 and January 2026 onward) and unzips it into one folder of CSV files per month. Every run also checks the last three months for newly published archives, so new months arrive without code changes. A month that is already complete is skipped, so a run with nothing new takes minutes. When the download finishes, it starts `citibike-prepare`. | `YYYYMM-citibike-tripdata/*.csv` in the volume |
| `external-data-download` | `jobs/external_data_download.py` | Daily 05:00 New York time | Collects every non-trip source: MTA subway alerts for 2026 and monthly G delay incidents (re-downloaded every day), the live MTA alert feed and the Citi Bike GBFS station feeds (a timestamped copy is archived every day, which builds the history the live feeds do not keep), and one-time downloads of the MTA subway stations, the 2012 to 2020 G alerts, hourly ridership at the G stations and the historical weather. Hourly weather for the Greenpoint grid cell is extended every day. Weather is requested in UTC and converted to New York time so that daylight saving time is handled correctly. | MTA, weather and GBFS files in the volume |
| `citibike-prepare` | `jobs/citibike_prepare.py` | After every `citibike-download` run, or started by hand | Cleans the raw files and builds all the Delta tables listed in part C. Trip months are loaded incrementally: a month is read only when it is new or its files changed, and the other tables are rebuilt in a few minutes. It records every quality check in the `data_quality_checks` table. | 28 Delta tables in `workspace.citibike` |

Two inputs are not downloaded by a job: the historical station status snapshots (`station_status/`, an export from a GBFS poller, August 21 to October 1, 2026) and the station information snapshots (`station_info.parquet`). They were uploaded to the volume once. The MTA Weekender closure windows are entered by hand from the MTA notices and kept in `src/citibike/study.py` (Section 2.2.1).

### B. Cleaning rules applied by `citibike-prepare`

The rules below are the same ones described in Section 2.3. The job applies them once to all the data, and records how many rows each rule affected.

| Data | Cleaning and preparation | Where to check the result |
|---|---|---|
| Trips (63.5 million rows read from 15 months of CSV files) | All fields are read as text and then typed. Station codes stay text, so codes such as `4665.02` keep their exact form. Times are parsed as New York clock time. Trips with a missing or unreadable start or end time, or an end time at or before the start, are removed. Repeated `ride_id` values are removed. Trips without a start or end station are kept, because the other end can still be counted. A trip duration in minutes is added. | `trips_ingest_log`: rows read, missing stations, invalid times, nonpositive durations, duplicates and rows kept for each month. In the current data no row failed the time, duration or duplicate rules. |
| Hourly demand | Departures are counted by start station and start hour, and arrivals by end station and end hour. | `trips_station_hourly` |
| Station status (29.6 million snapshots) | Fetch times are converted from UTC to New York time. Snapshots are summarized per station and hour: mean bikes and docks available, minimum open docks, number of snapshots with no bikes or no docks, and number of snapshots. | `station_status_hourly`; checks for missing values and negative counts are in `data_quality_checks` (none found) |
| Stations | The latest daily snapshot is used for location and capacity. The station information field `short_name` links the trip station codes to the GBFS `station_id` used by the status data. | `stations` |
| Weather | The two Open-Meteo files are combined into one hourly series for the Greenpoint grid cell, which serves all 43 Greenpoint stations. | `weather_hourly` |
| MTA files | Values are kept unchanged; only dates are parsed and column names are put in snake case. The G stops are selected from the subway stations file. | `raw_*` tables, `mta_g_stops` |

After cleaning, the job builds the study tables exactly as described in Sections 2.2 to 2.6, with the functions in `src/citibike/transformations.py`. This notebook uses the same functions where it rebuilds small lookups, so the job and the notebook cannot drift apart.

### C. Tables in the catalog

All tables are Delta tables in the schema `workspace.citibike`, next to the volume `raw` that holds the downloaded files. Row counts are as of November 21, 2026; the trip, weather and alert tables grow as the jobs add data, while the analysis panels cover the fixed study periods. The cell after the notebook setup lists the current tables and row counts.

**Raw tables.** One table per reference file, with the values unchanged. They make every source available as a table, including from a local editor.

| Table | Rows | Content |
|---|---:|---|
| `raw_station_info` | 108,047 | Daily station information snapshots (all snapshots of the export) |
| `raw_mta_subway_stations` | 496 | MTA subway stations file (all lines) |
| `raw_mta_subway_alerts_2026` | 30,794 | MTA subway alert updates for 2026 |
| `raw_mta_g_alerts_2012_2020` | 2,640 | Historical alert updates that mention the G train |
| `raw_mta_g_delay_incidents_monthly` | 308 | Monthly G line delay incidents by category |
| `raw_mta_g_station_hourly_ridership_2026` | 36,514 | Hourly ridership at the 21 G station complexes, August to mid October 2026 |
| `raw_mta_live_alerts_snapshot` | 1 | The live MTA alert feed snapshot used in Section 2.1, stored as one JSON document |
| `raw_greenpoint_station_weather_lookup` | 43 | Greenpoint station to weather grid cell lookup |
| `raw_greenpoint_weather_hourly_2026` | 3,336 | Hourly weather for 2026, extended daily |
| `raw_greenpoint_weather_hourly_extended` | 8,016 | Hourly weather for May to September 2024 and February to July 2026 |

**Cleaned tables.** General purpose tables that any analysis can start from.

| Table | Rows | Grain | Content |
|---|---:|---|---|
| `trips` | 63,473,804 | one trip | Cleaned trips with typed times and coordinates, duration and the source month (`trip_month`), May 2024 to October 2026 |
| `trips_ingest_log` | 15 | trip month | Source files, rows read, rows removed by each rule and rows kept |
| `trips_station_hourly` | 15,658,468 | station code by hour | Departures and arrivals for every station and month (hours with at least one trip) |
| `station_status_hourly` | 2,472,520 | station by hour | Hourly availability summary for 2,527 stations, August 21 to October 1, 2026 |
| `stations` | 2,520 | station | Latest station snapshot with location, capacity, distance to the nearest G stop and study group |
| `weather_hourly` | 11,352 | hour | Temperature (degrees Celsius) and precipitation (mm) for the Greenpoint grid cell |
| `mta_g_stops` | 21 | G stop | G train stops with coordinates and the affected segment flag |

**Analysis tables.** The study specific tables built in Sections 2.2 to 2.6 and used in Sections 3 to 6.

| Table | Rows | Built in | Used for |
|---|---:|---|---|
| `g_service_closures_confirmed` | 4 | 2.2.1 | The August and September 2026 G closure weekends (220 closure hours) |
| `g_service_closures_confirmed_pre_august` | 9 | 2.2.1 | The earlier 2026 closure weekends |
| `g_service_reduction_windows_inferred` | 8 | 2.1 | Closure windows inferred from G ridership, a check on the notices |
| `greenpoint_station_exposure` | 43 | 2.5 | Distance from each Greenpoint station to the nearest affected stop and its exposure band |
| `analysis_station_hour_panel` | 62,952 | 2.5 | Greenpoint stations by hour, August and September 2026, the only panel with station availability |
| `model_station_sets` | 154 | 2.6 | The 94 corridor and 60 control stations |
| `mta_subway_shutdowns` | 118 | 2.6 | One row per shutdown event and affected G stop (16 events) |
| `analysis_station_hour_panel_extended` | 1,485,072 | 2.6 | **Main modeling table**: 153 stations by hour for May to September 2024 and January to September 2026, with outcomes, weather, calendar, closure flags, baseline, profile, status proxy and the train, validation and test split |
| `baseline_accuracy_by_month` | 24 | 2.6 | Error of the four week baseline, the reference every model must beat |
| `shutdown_effect_screening` | 10 | 2.6 | Actual to normal demand at exposed and control stations for each event type (Figure 10) |
| `data_quality_checks` | 36 | all | Every check recorded by the latest `citibike-prepare` run |

Sections 3 to 6 start from `analysis_station_hour_panel_extended` (models and evaluation) and `analysis_station_hour_panel` (station availability). Section 3.8 describes the columns of the extended panel and the rules to follow before modeling.

### D. How this notebook reads the data

The setup cell below connects to Databricks and defines one helper, `read_table(name, columns, where)`. The helper sends the query to Databricks, applies any column selection and filter there, and returns only the result as a pandas table. Large aggregations, such as counting hourly trips from the 63 million trip rows, also run in Databricks (`data_access.hourly_station_flows`). As a safeguard, a query that would return more than 3 million rows stops with an error instead of downloading a whole table.

The notebook runs in two places with the same code:

* **In Databricks**, it uses the workspace's own Spark session, and the catalog, schema and volume can be changed with the widgets at the top of the notebook.
* **In VS Code**, Databricks Connect sends the same queries to serverless compute in the workspace, while pandas, the figures and the models run on the local computer. The schema can be changed with the `CITIBIKE_CATALOG`, `CITIBIKE_SCHEMA` and `CITIBIKE_VOLUME` environment variables. Setup steps are in the repository README.

No cell in this notebook downloads, unzips or writes data. If a table is missing, the notebook stops with a "table not found" error; running the `citibike-prepare` job builds it.

## 2.1 Data sources and provenance

The project uses the following data sources. All of them are stored in the Databricks volume and loaded into the tables described above. External sources are requirements for a later event comparison, not data already analyzed.

1. **Citi Bike trip history.** Monthly trip records are stored in monthly archive folders for May to September 2024 and January to September 2026, which the `citibike-download` job downloads and unzips, adding newly published months automatically. The analysis uses trip start time, end time, start station identifier, and end station identifier.
2. **Historical Citi Bike station status.** The local Parquet snapshots were supplied as an NYC export from BigQuery and contain a fetch timestamp, canonical station identifier, reported available bikes, and reported open docks. Snapshot times are recorded in UTC and converted to New York local time for analysis. The file metadata is the provenance record: the snapshots come from a GBFS status poller (poller version 1.2, run manually), were written by a Spark export job on October 1, 2026, and are partitioned by `nyc_date` from August 21 to October 1, 2026. The first usable snapshot hour is August 21, 17:00 New York time.
3. **Citi Bike station information.** Local dated snapshots contain canonical station identifiers, short station identifiers, station names, coordinates, and capacity. The short identifier provides the crosswalk from trip history to station status.
4. **Weather.** Hourly temperature and precipitation are retrieved from the Open Meteo historical archive for May to September 2024 and January to September 2026, which covers all trip months and the full station status window. The station to weather grid mapping follows the method in the Citi Bike data pipeline reference. The project proposal also identifies BigQuery weather as a possible source; this notebook uses the documented API source because no local BigQuery weather table was supplied.
5. **MTA service alerts.** Subway alert updates are retrieved from the New York State Open Data API. The API record date is the notice time, not necessarily the actual service interruption time. The available archive currently ends on August 11, 2026. The live MTA alert feed is also read as a snapshot and parsed into G service event periods with affected stops and start and end times; it only lists work from October 19, 2026 onward, so neither alert source gives closure hours inside the station status period. Closure intervals therefore come from the MTA Weekender planned service notices (Section 2.2.1), which are cross checked against hourly ridership at G stations. Monthly G line delay causing incident counts are collected as context.
6. **MTA subway stops.** G train stop names and coordinates are read from the local MTA Subway Stations file (`MTA_Subway_Stations_20261003.csv`, 496 stops, 21 with G service), exported from New York State Open Data. These coordinates support distance based station exposure after event alerts have been validated.

The monthly trip files, station files, weather, alert and stop data are collected by the `citibike-download` and `external-data-download` Databricks jobs and turned into tables by `citibike-prepare` (see the overview above); this notebook does not download or unzip anything. Historical station status was obtained from BigQuery and uploaded once to `datasource/station_status` in the volume, together with `station_info.parquet`. The related New York BigQuery tables are `bigquery-public-data.new_york_citibike.citibike_stations` and `citibike_trips`; the status files are an NYC export partitioned by `nyc_date`, and the export job metadata is recorded in the data sources document (`docs/data_sources.md`). API URLs, query filters, coverage, units, and limits are recorded in the corresponding code and the data sources document (`docs/data_sources.md`).

In [ ]:
%pip install -q lightgbm


### Notebook setup

The next cell makes the project package `src/citibike` importable, reads the catalog, schema and volume settings, opens the Spark session and checks that every trip month used by the study has been loaded. The cell after it lists the tables currently in the catalog.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable: walk up from this notebook to the folder that holds databricks.yml.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "databricks.yml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"Could not find the project root (databricks.yml) above {Path.cwd()}")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
from citibike import data_access, study
from citibike import transformations as tf
from citibike.config import load_config

# Catalog, schema and volume: notebook widgets on Databricks, CITIBIKE_* environment variables or defaults elsewhere
CONFIG = load_config(globals().get("dbutils"))
spark = data_access.get_spark()


def read_table(name, columns=None, where=None, max_rows=data_access.DEFAULT_MAX_ROWS):
    """Load a project table as pandas. Filters run in Databricks before any rows are transferred."""
    return data_access.read_pandas(name, columns=columns, where=where, max_rows=max_rows, spark=spark, config=CONFIG)


# Trip file months used in Section 2.1 (January, August and September 2026) and the extra months of Section 2.6
TRIP_MONTHS = tuple(month.replace("-", "") for month in study.TRIP_MONTHS)
TRIP_MONTHS_EXTENDED = tuple(month.replace("-", "") for month in study.EXTENDED_MONTHS if month not in study.TRIP_MONTHS)

ingest_log = read_table("trips_ingest_log").sort_values("trip_month").reset_index(drop=True)
missing_months = sorted(set(study.EXTENDED_MONTHS) - set(ingest_log["trip_month"]))
if missing_months:
    raise LookupError(f"Trip months {missing_months} are not in {CONFIG.catalog}.{CONFIG.schema}.trips. "
                      "Run the citibike-download job (it starts citibike-prepare) first.")
print("Tables:", f"{CONFIG.catalog}.{CONFIG.schema}")
print(f"Trip months ready: {len(study.EXTENDED_MONTHS)}")

In [ ]:
# Tables in the project schema with their description and current row count
table_list = spark.sql(f"""
    SELECT table_name, comment
    FROM `{CONFIG.catalog}`.information_schema.tables
    WHERE table_schema = '{CONFIG.schema}'
    ORDER BY table_name
""").toPandas()
table_list.insert(1, "rows", [
    spark.table(f"`{CONFIG.catalog}`.`{CONFIG.schema}`.`{name}`").count() for name in table_list["table_name"]
])
table_list["layer"] = table_list["table_name"].map(
    lambda name: "raw" if name.startswith("raw_") else
    "cleaned" if name in {"trips", "trips_ingest_log", "trips_station_hourly", "station_status_hourly", "stations",
                          "weather_hourly", "mta_g_stops"} else "analysis"
)
print(f"{len(table_list)} tables in {CONFIG.catalog}.{CONFIG.schema}")
display(table_list.sort_values(["layer", "table_name"])[["layer", "table_name", "rows", "comment"]].reset_index(drop=True))

In [ ]:
# What the jobs have delivered: trip files loaded into the trips table and rows in the station tables
inventory = {
    **{f"{month} trip CSV files loaded": int(ingest_log.loc[ingest_log["trip_month"].eq(month), "csv_files"].sum())
       for month in study.TRIP_MONTHS},
}
for name in ("station_status_hourly", "raw_station_info", "stations", "raw_mta_live_alerts_snapshot"):
    inventory[f"{name} rows"] = spark.table(CONFIG.table(name)).count()
for source, value in inventory.items():
    print(f"{source}: {value}")

**Inventory note:** This cell counts what the Databricks jobs delivered: the trip files loaded into the `trips` table and the rows in the station tables. The next cells load the trip and station data, then the weather and MTA tables. A missing table stops the notebook with a "table not found" error; run the `citibike-prepare` job to build it. Review the source dates and coverage before interpreting the loaded data.

In [ ]:
# Latest station information snapshot and the Greenpoint screening rectangle
station_info, latest_snapshot = tf.latest_station_snapshot(read_table("raw_station_info"))
greenpoint_stations = tf.greenpoint_screen(station_info)
greenpoint_ids = greenpoint_stations["station_id"].dropna().unique()
greenpoint_trip_ids = greenpoint_stations["trip_station_id"].dropna().unique()

# Hourly departures and arrivals at the Greenpoint stations from the January, August and September 2026 trip files,
# counted in Databricks from the cleaned trips table (trips with a missing time or a nonpositive duration were dropped
# by the citibike-prepare job).
trip_demand = data_access.hourly_station_flows(
    study.TRIP_MONTHS, greenpoint_trip_ids, spark=spark, config=CONFIG
).rename(columns={"short_name": "station_id"})
trip_demand["station_id"] = trip_demand["station_id"].astype("string")
trip_station_ids = data_access.trip_station_codes(study.TRIP_MONTHS, spark=spark, config=CONFIG)

# Source checks recorded by the citibike-prepare job for the same trip files
months_log = ingest_log[ingest_log["trip_month"].isin(study.TRIP_MONTHS)]
trip_file_count = int(months_log["csv_files"].sum())
trip_rows = int(months_log["rows_read"].sum())
missing_station_ids = {"departures": int(months_log["missing_start_station"].sum()),
                       "arrivals": int(months_log["missing_end_station"].sum())}
# The job counts a trip with a missing or unreadable start or end time once, for both flows
missing_trip_times = {flow: int(months_log["invalid_time"].sum()) for flow in ("departures", "arrivals")}
invalid_trip_times = {"departures": 0, "arrivals": 0}
invalid_trip_duration_rows = int(months_log["nonpositive_duration"].sum())

# Hourly station status for the Greenpoint stations; the job converted the fetch times from UTC to New York time
id_list = ", ".join(f"'{station_id}'" for station_id in greenpoint_ids)
station_status_hourly = read_table("station_status_hourly", where=f"station_id IN ({id_list})")
station_status_hourly["station_id"] = station_status_hourly["station_id"].astype("string")
station_status_hourly = station_status_hourly.sort_values(["station_id", "hour"]).reset_index(drop=True)
status_source_rows = int(station_status_hourly["snapshot_count"].sum())
job_checks = read_table("data_quality_checks").set_index("check")["result"]
status_missing_values = {"any required field, all stations": int(job_checks["Status snapshots with a missing required value"].replace(",", ""))}
status_negative_count_rows = int(job_checks["Status snapshots with negative bike or dock counts"].replace(",", ""))

print(f"Trip files loaded: {trip_file_count}")
print(f"Trip rows read: {trip_rows:,}")
print(f"Greenpoint station hours with trip records: {len(trip_demand):,}")
print(f"Greenpoint station hours with status records: {len(station_status_hourly):,}")
print(f"Station information records in latest snapshot: {len(station_info):,}")
print(f"Missing station identifiers skipped: {missing_station_ids}")
print(f"Missing trip timestamps skipped: {missing_trip_times}")
print(f"Invalid trip timestamps skipped: {invalid_trip_times}")
print(f"Trips with nonpositive recorded duration excluded: {invalid_trip_duration_rows:,}")
print(f"Greenpoint station status snapshots read: {status_source_rows:,}")
print(f"Status rows with negative bike or dock counts (all stations): {status_negative_count_rows:,}")
print(f"Missing values in required status fields: {status_missing_values}")

In [ ]:
import json

mta_subway_alerts = read_table("raw_mta_subway_alerts_2026")
mta_g_stops = read_table("mta_g_stops").sort_values(["stop_name", "gtfs_stop_id"]).reset_index(drop=True)
station_weather_lookup = read_table("raw_greenpoint_station_weather_lookup")
greenpoint_weather = read_table("raw_greenpoint_weather_hourly_2026").sort_values("time").reset_index(drop=True)

print(f"MTA subway alert updates collected: {len(mta_subway_alerts):,}")
print(f"MTA alert update coverage: {mta_subway_alerts['date'].min()} to {mta_subway_alerts['date'].max()}")
print(f"G train stops with coordinates: {len(mta_g_stops):,}")
print(f"Greenpoint stations mapped to weather grid cells: {station_weather_lookup['station_id'].nunique():,}")
print(f"Distinct weather grid cells: {greenpoint_weather['weather_cell_id'].nunique():,}")
print(f"Hourly weather coverage: {greenpoint_weather['time'].min()} to {greenpoint_weather['time'].max()}")
display(greenpoint_weather.groupby("month")[["time", "temperature_2m", "precipitation"]].agg(
    {"time": "count", "temperature_2m": "count", "precipitation": "count"}
).rename(columns={"time": "hourly rows", "temperature_2m": "temperature values", "precipitation": "precipitation values"}))

In [ ]:
# Hourly weather for the extended trip months (2024 shutdown check and 2026 baseline)
extended_weather = read_table("raw_greenpoint_weather_hourly_extended").sort_values("time").reset_index(drop=True)
print(
    f"Extended weather rows: {len(extended_weather):,}; "
    f"missing values: {int(extended_weather[['temperature_2m', 'precipitation']].isna().sum().sum())}"
)

In [ ]:
import numpy as np

QUARTER_MILE_KM = study.QUARTER_MILE_KM
haversine_km = tf.haversine_km

# Station to G stop distances and the quarter mile exposure rule
stop_pairs = greenpoint_stations[["station_id", "lat", "lon"]].drop_duplicates("station_id").merge(
    mta_g_stops[["gtfs_stop_id", "stop_name", "gtfs_latitude", "gtfs_longitude"]], how="cross"
)
stop_pairs["distance_km"] = haversine_km(
    stop_pairs["lat"], stop_pairs["lon"], stop_pairs["gtfs_latitude"], stop_pairs["gtfs_longitude"]
)
stop_pairs["within_quarter_mile"] = stop_pairs["distance_km"] <= QUARTER_MILE_KM
exposed_stop_ids = set(stop_pairs.loc[stop_pairs["within_quarter_mile"], "gtfs_stop_id"])
exposed_station_count = stop_pairs.loc[stop_pairs["within_quarter_mile"], "station_id"].nunique()

# Live MTA alert snapshot, parsed into one row per G service event and active period
live_alerts = json.loads(read_table("raw_mta_live_alerts_snapshot")["snapshot_json"].iloc[0])

event_rows = []
for entity in live_alerts["entity"]:
    alert = entity["alert"]
    informed = alert.get("informed_entity", [])
    if not any(item.get("route_id") == "G" for item in informed):
        continue
    stops = sorted({item["stop_id"] for item in informed if item.get("route_id") == "G" and item.get("stop_id")})
    translations = alert.get("header_text", {}).get("translation", [{}])
    mercury = alert.get("transit_realtime.mercury_alert", {})
    for period in alert.get("active_period", []):
        event_rows.append({
            "alert_id": entity["id"],
            "alert_type": mercury.get("alert_type"),
            "start_utc": pd.to_datetime(period["start"], unit="s", utc=True),
            "end_utc": pd.to_datetime(period["end"], unit="s", utc=True) if period.get("end") else pd.NaT,
            "g_stop_count": len(stops),
            "g_stop_ids": ";".join(stops),
            "touches_greenpoint_stops": bool(exposed_stop_ids.intersection(stops)),
            "header": translations[0].get("text"),
        })
g_service_events = pd.DataFrame(event_rows).sort_values("start_utc").reset_index(drop=True)
feed_time = pd.to_datetime(live_alerts["header"]["timestamp"], unit="s", utc=True)

# Monthly G line delay causing incidents (open data, covers the status window)
g_delay_incidents = read_table("raw_mta_g_delay_incidents_monthly")

status_window = (pd.Timestamp("2026-08-21", tz="UTC"), pd.Timestamp("2026-10-02", tz="UTC"))
overlap = g_service_events[
    (g_service_events["start_utc"] < status_window[1])
    & (g_service_events["end_utc"].fillna(pd.Timestamp.max.tz_localize("UTC")) > status_window[0])
]
print(f"Greenpoint stations within a quarter mile of a G stop: {exposed_station_count} of {stop_pairs['station_id'].nunique()}")
print(f"Live feed timestamp: {feed_time}")
print(f"G service event periods in live feed: {len(g_service_events):,} ({g_service_events['start_utc'].min()} to {g_service_events['end_utc'].max()})")
print(f"Event periods touching Greenpoint stops: {int(g_service_events['touches_greenpoint_stops'].sum()):,}")
print(f"Event periods overlapping the status window: {len(overlap):,}")
display(g_service_events.groupby("alert_type").size().rename("event periods").to_frame())
display(g_delay_incidents.groupby(g_delay_incidents["month"].dt.to_period("M"))["incidents"].sum().tail(10).rename("G delay incidents").to_frame())

**Historical alert archive.** The MTA Service Alerts 2012 to 2020 dataset uses the agency label Subway and ends on March 31, 2020. It cannot overlap the 2026 station status window, so it is not used to label 2026 hours. It supplies 2,640 G train alert updates (delays, service changes, part suspensions and suspensions) that can be used to test the alert parser, to build a hand checked calendar of past G suspensions, and to plan a comparison with older Citi Bike trip archives. Alert dates are publication times, not confirmed closure times.


In [ ]:
legacy_g_alerts = read_table("raw_mta_g_alerts_2012_2020")

print(f"Legacy G train alert updates (2012 to 2020): {len(legacy_g_alerts):,}")
print(f"Coverage: {legacy_g_alerts['date'].min()} to {legacy_g_alerts['date'].max()}")
display(legacy_g_alerts.groupby("status_label").size().sort_values(ascending=False).head(8).rename("alert updates").to_frame())
display(legacy_g_alerts.groupby(legacy_g_alerts["date"].dt.year).size().rename("alert updates").to_frame())

### Ridership check of the G service reduction

No machine readable alert source covers closure hours inside the August 21 to October 1 status window, so ridership serves as an independent check on the planned service notices in Section 2.2.1. The MTA hourly ridership dataset does cover it, and a part suspension of the G through Greenpoint should remove most riders at the two G only stations in the neighborhood, Greenpoint Av and Nassau Av. Hourly ridership for 21 G station complexes is therefore pulled from the MTA hourly ridership dataset and cached. For each station, ridership is divided by the median ridership for the same weekday and clock hour across August 1 to October 14. An hour is flagged when both stations fall below 0.35 of that typical level, the hour is 06:00 or later, and the date is not in the Labor Day holiday period (September 4, 7 and 8). Runs of at least three flagged hours are kept, and runs separated by two hours or less are merged.


In [ ]:
import matplotlib.pyplot as plt

G_COMPLEX_IDS = study.G_COMPLEX_IDS
g_ridership = read_table("raw_mta_g_station_hourly_ridership_2026")

# Windows where both detector stations keep under 35 percent of their typical ridership for 3+ hours
# (src/citibike/transformations.py; the citibike-prepare job saves the same windows as g_service_reduction_windows_inferred)
ridership_check = tf.infer_reduction_windows(g_ridership)
g_closure_windows = ridership_check.windows
ridership_wide = ridership_check.ridership_wide
ridership_ratio = ridership_check.ridership_ratio
detector_stations = study.DETECTOR_STATIONS
hour_keys = [ridership_wide.index.dayofweek, ridership_wide.index.hour]

print(f"Hourly rows for G station complexes: {len(g_ridership):,}")
print(f"Inferred G service reduction windows: {len(g_closure_windows)} "
      f"({int(g_closure_windows['in_status_window'].sum())} inside the station status period)")
display(g_closure_windows.assign(median_ratio=g_closure_windows["median_ratio"].round(2)))

fig, ax = plt.subplots(figsize=(12, 4.5))
visible = ridership_ratio.loc["2026-08-21":"2026-10-01 23:00"]
for station in detector_stations:
    ax.plot(visible.index, visible[station], linewidth=0.8, label=station)
for _, window in g_closure_windows[g_closure_windows["in_status_window"]].iterrows():
    ax.axvspan(window["start_local"], window["end_local"], color="tab:red", alpha=0.2)
ax.axhline(0.35, color="black", linestyle="--", linewidth=0.8, label="Detection threshold (0.35)")
ax.set_ylabel("Ridership relative to typical weekday hour")
ax.set_xlabel("Local date")
ax.set_title("Hourly ridership at Greenpoint G stations with inferred service reduction windows")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Figure interpretation.** Each line is hourly ridership divided by the typical level for that weekday and hour, so a value of 1 means a normal hour. Both lines stay near 1 on most days and drop close to 0 together during the shaded red windows, which fall on weekends. A common drop at two stations on the same hours is what a service suspension produces, while a station specific issue or a holiday would not move both lines in this way. These windows are inferred from rider counts, not read from an MTA notice, so they are a proxy for closure hours. The start and end times are approximate to the hour, and low ridership caused by weather or a holiday could also be flagged. They should be checked against the MTA planned work notices before they are used as the exposure label.


### 2.2.1 Confirmed G closure windows from MTA planned service notices

The MTA Weekender notices for August 7 to 10, August 21 to 24, September 11 to 14 and September 25 to 28 each state that G trains do not run between Court Sq and Bedford-Nostrand Avs from 9:45 p.m. Friday to 5 a.m. Monday, because of signal modernization, with free shuttle buses between those stations. The weekends of August 14, August 28 and September 18 do not list this work, and the Labor Day weekend notice lists no G work either. These notices replace the ridership inferred windows as the closure definition for the analysis. The ridership windows are kept as an independent check: if the notices are right, the inferred windows should fall inside them. The notices also list stops skipped on the southern F and G section on some weekends, which are not treated as closures here.


The same notices were read for every weekend from January to July 2026 so that the pre August closure weekends can serve as a longer history for the baseline and for placebo checks. Nine of them state the same Court Sq to Bedford-Nostrand Avs suspension: February 27, March 6, March 13, March 20, March 27, April 17, June 5, June 19 and July 10. Together with the four weekends above, that gives 13 confirmed weekends in 2026. The January 2 to 9, 16 and 23 weekends, the February 6, 13 and 20 weekends, April 3, 10 and 24, May 1, 8, 15, 22 and 29, June 12 and 26, and July 3, 17, 24 and 31 weekends do not list it. The February 27 notice states the 9:45 p.m. Friday to 5 a.m. Monday times in full. For the other pre August weekends only the affected segment was read from the notice summary, and the same times are applied, so the `times_assumed` column marks them.

In [ ]:
# MTA Weekender notices (src/citibike/study.py): no G trains between Court Sq and Bedford-Nostrand Avs from
# 9:45 p.m. Friday to 5 a.m. Monday. The citibike-prepare job turns them into g_service_closures_confirmed.
MTA_WEEKENDER_NOTICES = study.MTA_WEEKENDER_NOTICES
g_confirmed_windows = read_table("g_service_closures_confirmed").sort_values("start_local").reset_index(drop=True)

# Whole clock hours fully inside a window: 22:00 Friday through 04:00 Monday
closure_hours = tf.closure_hour_index(g_confirmed_windows)

inferred_hours = pd.DatetimeIndex(np.concatenate([
    pd.date_range(row.start_local, row.end_local - pd.Timedelta(hours=1), freq="h").to_numpy()
    for row in g_closure_windows.itertuples()
]))
agreement = inferred_hours.isin(closure_hours).mean()
print(f"Confirmed closure hours: {len(closure_hours):,} across {len(g_confirmed_windows)} weekends")
print(f"Share of ridership inferred hours that fall inside a confirmed window: {agreement:.1%}")
print(f"Confirmed weekends with at least one inferred window: "
      f"{sum(any((w.start_local <= i_start < w.end_local) for i_start in g_closure_windows['start_local']) for w in g_confirmed_windows.itertuples())} of {len(g_confirmed_windows)}")
display(g_confirmed_windows.drop(columns="notice_url"))

In [ ]:
# The nine earlier 2026 closure weekends (src/citibike/study.py); times_assumed marks notices whose times were inferred
PRE_AUGUST_NOTICES = study.PRE_AUGUST_NOTICES
g_confirmed_pre_august = read_table("g_service_closures_confirmed_pre_august").sort_values("start_local").reset_index(drop=True)
print(f"Confirmed G closure weekends before August 2026: {len(g_confirmed_pre_august)}")
print(f"Confirmed G closure weekends in 2026 in total: {len(g_confirmed_pre_august) + len(g_confirmed_windows)}")

**Interpretation of the closure check.** The four notices define 220 closure hours (55 per weekend, from 22:00 Friday to 04:00 Monday). Every one of the ridership inferred hours falls inside a confirmed window, and each confirmed weekend contains at least one inferred window. Two independent sources therefore agree on when G service was cut. The confirmed windows are longer because the ridership rule only flags daytime hours from 06:00 and needs both Greenpoint Av and Nassau Av to be far below normal, while overnight ridership is too small to separate from usual quiet hours. The confirmed windows are used as the closure indicator in the panel. The August 7 to 10 weekend lies before the station status period, so it supports trip and weather analysis but has no availability data.

**Loaded data:** The code loads hourly departure and arrival counts for the January, August and September trip files, counted in Databricks from the cleaned `trips` table. It also loads the latest station information snapshot and the hourly station status summary built by the `citibike-prepare` job. Trip timestamps are kept in local time; status timestamps were converted from UTC to New York time before aggregation. The hourly tables remain separate so differences in their time coverage are visible before they are joined. Station attributes come from the latest snapshot and may differ from historical station locations or capacity.

In [ ]:
import pandas as pd

GREENPOINT_LATITUDE = (40.720, 40.740)
GREENPOINT_LONGITUDE = (-73.965, -73.935)

greenpoint_stations = station_info.loc[
    station_info["lat"].between(*GREENPOINT_LATITUDE)
    & station_info["lon"].between(*GREENPOINT_LONGITUDE)
    & station_info["capacity"].gt(0)
].copy()
greenpoint_stations["trip_station_id"] = greenpoint_stations["short_name"].astype("string")
greenpoint_ids = greenpoint_stations["station_id"].dropna().unique()
greenpoint_trip_ids = greenpoint_stations["trip_station_id"].dropna().unique()

greenpoint_trips = trip_demand.rename(
    columns={"station_id": "trip_station_id"}
).loc[lambda frame: frame["trip_station_id"].isin(greenpoint_trip_ids)].merge(
    greenpoint_stations[["station_id", "trip_station_id", "name", "lat", "lon", "capacity"]],
    on="trip_station_id",
    how="left",
    validate="many_to_one",
)
greenpoint_trips["date"] = greenpoint_trips["hour"].dt.normalize()
greenpoint_trips["hour_of_day"] = greenpoint_trips["hour"].dt.hour
greenpoint_trips["weekday"] = greenpoint_trips["hour"].dt.day_name()
greenpoint_trips["day_type"] = greenpoint_trips["hour"].dt.dayofweek.map(
    lambda day: "Weekend" if day >= 5 else "Weekday"
)
greenpoint_trips["month"] = greenpoint_trips["hour"].dt.strftime("%Y-%m")

greenpoint_status = station_status_hourly.loc[
    station_status_hourly["station_id"].isin(greenpoint_ids)
].merge(
    greenpoint_stations[["station_id", "name", "lat", "lon", "capacity"]],
    on="station_id",
    how="left",
    validate="many_to_one",
)
greenpoint_status["hour_of_day"] = greenpoint_status["hour"].dt.hour
greenpoint_status["weekday"] = greenpoint_status["hour"].dt.day_name()
greenpoint_status["day_type"] = greenpoint_status["hour"].dt.dayofweek.map(
    lambda day: "Weekend" if day >= 5 else "Weekday"
)

trip_hours_by_month = (
    trip_demand.assign(month=trip_demand["hour"].dt.strftime("%Y-%m"))
    .groupby("month")[["departures", "arrivals"]]
    .sum()
)
trip_date_min = trip_demand["hour"].min()
trip_date_max = trip_demand["hour"].max()
status_date_min = station_status_hourly["hour"].min()
status_date_max = station_status_hourly["hour"].max()
current_trip_station_ids = set(station_info["short_name"].dropna())
unmatched_trip_stations = trip_station_ids - current_trip_station_ids
duplicate_trip_hour_keys = int(trip_demand.duplicated(["station_id", "hour"]).sum())
duplicate_status_hour_keys = int(station_status_hourly.duplicated(["station_id", "hour"]).sum())
duplicate_current_station_ids = int(station_info["station_id"].duplicated().sum())
trip_status_join = greenpoint_trips.merge(
    greenpoint_status[
        ["station_id", "hour", "bikes_available_mean", "docks_available_mean",
         "zero_bike_snapshots", "zero_dock_snapshots", "snapshot_count"]
    ],
    on=["station_id", "hour"],
    how="outer",
    indicator=True,
    validate="one_to_one",
)

quality_summary = pd.DataFrame(
    [
        ("Trip CSV files read", trip_file_count),
        ("Raw trip rows read", trip_rows),
        ("Departure records without a station ID", missing_station_ids["departures"]),
        ("Arrival records without a station ID", missing_station_ids["arrivals"]),
        ("Departure timestamps missing or invalid", missing_trip_times["departures"] + invalid_trip_times["departures"]),
        ("Arrival timestamps missing or invalid", missing_trip_times["arrivals"] + invalid_trip_times["arrivals"]),
        ("Trips with nonpositive recorded duration excluded", invalid_trip_duration_rows),
        ("Greenpoint station status snapshots read", status_source_rows),
        ("Missing values in required status fields", sum(status_missing_values.values())),
        ("Status rows with negative bike or dock counts", status_negative_count_rows),
        ("Latest station information snapshot", str(latest_snapshot)),
        ("Duplicate station IDs in latest station snapshot", duplicate_current_station_ids),
        ("Duplicate trip station hour keys", duplicate_trip_hour_keys),
        ("Duplicate status station hour keys", duplicate_status_hour_keys),
        ("Stations in Greenpoint screening rectangle", len(greenpoint_stations)),
        ("Trip station IDs absent from latest station short name crosswalk", len(unmatched_trip_stations)),
        ("Greenpoint trip station hours within status date range", int(greenpoint_trips["hour"].between(status_date_min, status_date_max).sum())),
    ],
    columns=["Check", "Result"],
)

print(f"Greenpoint trip time coverage: {trip_date_min} to {trip_date_max}")
print(f"Greenpoint status time coverage: {status_date_min} to {status_date_max}")
print(f"Greenpoint station screen: {len(greenpoint_stations)} stations")
print(f"Greenpoint trip station hours: {len(greenpoint_trips):,}")
print(f"Greenpoint status station hours: {len(greenpoint_status):,}")
print("\nData quality summary")
display(quality_summary)
print("\nTrip activity by calendar month")
display(trip_hours_by_month)
print("\nGreenpoint trip and status key coverage")
display(trip_status_join["_merge"].value_counts().rename_axis("source coverage").to_frame("station hours"))

**Data preparation results:** The table above records the current file and quality checks. Records without a station identifier cannot be assigned to a station, so they are excluded from station level counts and reported separately. The spatial screen uses a rectangle and current station information; it is suitable for initial description, not for deciding whether a station was affected by a particular closure. The join coverage table compares trip and status station hours. A missing side of the join means that the source has no matching record for that key; it is not a zero count.

The August and September trip records overlap the station status records from August 21. Weather observations, MTA subway alert updates, and G train stop coordinates are collected from the APIs described in Section 2.2. The alert archive ends on August 11, 2026, while historical station status begins on August 21, 2026. Therefore, alert history does not identify G train closure intervals during the station status period. Those intervals come from the MTA Weekender notices in Section 2.2.1 and are checked against ridership. The figures below describe routine patterns and station inventory; they do not estimate the effect of a subway closure.

## 2.2 Collection and data understanding

The `citibike-download` job stores the trip archives in the Databricks volume and extracts them into month folders; the `citibike-prepare` job loads the extracted CSV files into the cleaned `trips` table, which this notebook reads. Historical station status is stored in date partitioned Parquet files. Station information is stored as dated snapshots. The current GBFS JSON files describe their fetch time and are not a substitute for historical status. The `external-data-download` job retrieves hourly Open Meteo archive weather, MTA subway alert updates, and MTA G train stop coordinates into the volume, and each file is available as a `raw_` table.

The trip loader reads every field of the CSV files in Spark and the hourly counts use four of them. Trip records use short station identifiers, while station status uses canonical GBFS identifiers. The station information field `short_name` provides the crosswalk. Trip times are treated as New York local time because the source values have no explicit time zone. Status fetch times are converted from UTC to New York local time. The weather request uses each screened station's coordinates with land cell selection, records the grid coordinates returned by Open Meteo, and reduces repeated station results to unique weather cell and hour records. This follows the station to grid mapping approach in the cited Citi Bike pipeline; its HRRR forecast configuration is not used for historical observations. The resulting station lookup allows each station to be associated with its grid cell. The original source files are left unchanged, and the hourly trip aggregation does not retain individual trip identifiers.

| Source fields | Meaning in the analysis |
| --- | --- |
| `started_at`, `ended_at` | Recorded trip start and end times. |
| `start_station_id`, `end_station_id` | Short identifiers used to assign departures and arrivals. |
| `station_id`, `short_name` | Canonical GBFS key and trip history crosswalk. |
| `name`, `lat`, `lon`, `capacity` | Station description and latest available location and capacity. |
| `fetched_at`, `num_bikes_available`, `num_docks_available` | Status snapshot time, reported bikes, and reported open docks. |
| `temperature_2m`, `precipitation` | Hourly weather at the mapped Open Meteo grid cell; temperature is degrees Celsius and precipitation is millimeters. |
| MTA `date`, `affected`, `header` | Alert update publication time and text fields; they require validation before defining event start or end times. |
| MTA stop coordinates | GTFS stop identifiers and coordinates used to calculate distance to G train stops. |
| Derived station and hour fields | Departure and arrival counts, mean inventory, zero inventory snapshot counts, and snapshot count. |

## 2.3 Data quality checks, results, and cleaning

The loader applies the following checks and transformations:

1. Read four required trip fields in chunks and keep station identifiers as strings so that identifiers with punctuation or leading zeros are preserved. Use the short name crosswalk to focus trip records on the 43 stations in the Greenpoint screening rectangle before parsing their timestamps.
2. Count rows missing a departure or arrival station identifier, then exclude those endpoint rows from station level aggregation. Missing timestamps and timestamps that cannot be parsed are counted separately.
3. Floor valid trip timestamps to the hour. The source timestamps have no explicit offset; the analysis treats the published clock time as New York local time. The convention was checked only for the loaded months. Daylight saving transition dates (March and November) fall in the modeled months and are assumed to be rare enough to ignore here, so they should be reviewed before modeling.
4. Convert the status fetch timestamps from UTC to New York local time. Retain hourly means, the minimum reported open docks, and the number of source snapshots used in each hourly summary.
5. Use the latest station information snapshot for the initial geographic screen. Historical station capacity and location may differ, so event analysis should use station attributes valid at the event date when available.
6. Exclude trips with both timestamps present when the end time is not later than the start time. Match trip station identifiers to the GBFS station identifier through the station information short name field. Match station status directly by GBFS station identifier rather than station name, and report trip identifiers absent from the crosswalk.

### Results of the checks

The checks were run on the seven local trip CSV files, historical status data, and external sources retrieved on October 24, 2026. The table reports measured outcomes and how each was handled.

| Check | Result | Cleaning decision or interpretation |
| --- | --- | --- |
| Trip records read | 6,799,246 | Read in chunks; station level measures were retained for stations in the Greenpoint screen. |
| Departure records without a start station ID | 4,251 | Cannot be assigned to a station; excluded from station level departures and counted in this quality report. |
| Arrival records without an end station ID | 23,375 | Cannot be assigned to a station; excluded from station level arrivals and counted in this quality report. |
| Missing or invalid trip timestamps | 0 | No timestamp rows were excluded for this reason. |
| Trips with end time at or before start time | 0 | No trips were excluded by the duration check. |
| Distinct trip station IDs absent from the latest station crosswalk | 117 | These IDs could not be matched to the latest station list. They are reported for review rather than matched by station name. |
| Screened station status snapshots with missing required values | 0 | No rows were excluded for missing station ID, fetch time, bike count, or dock count. |
| Screened station status snapshots with negative bike or dock counts | 0 | No count corrections or exclusions were required. |
| Duplicate station IDs in the latest station snapshot | 0 | The station information merge did not duplicate station records. |
| Duplicate station by hour keys after aggregation | 0 | Trip and status hourly tables each had unique station and hour keys. |
| MTA subway alert updates retrieved for 2026 | 30,794; latest update August 11 | These are notice updates, not verified service intervals. The archive ends before the station status sample begins. |
| G train stop records retrieved | 21 | Stop coordinates are available for later distance calculations; an alert must first identify which stops lost service. |
| Greenpoint stations linked to weather cells | 43 stations, 1 unique grid cell | The local screen maps all 43 stations to the same returned weather cell. This limits station level weather variation. |
| Hourly weather records | 2,208 for January, August and September combined | Temperature and precipitation are available for the 92 calendar days requested, in New York local time. |

These results support the hourly aggregation for the checks listed, but they do not establish that all source records are error free. The MTA alert date is the notice publication time, not necessarily the disruption start time, and this historical alert feed has no records overlapping the station status sample. This first pass does not assess duplicate ride identifiers, unusually long but positive trip durations, every source quality flag, or whether each station was active throughout the study period. Those checks require additional raw fields or dated station history and should be completed before modeling. Missing trip counts are not changed to zero; zeros are introduced only on the complete calendar grid used for monthly demand plots.

## 2.4 Integration and derived analysis table

Trip records are grouped by start station and start hour for departures, and by end station and end hour for arrivals. The two measures are aligned on station and hour. Status reports are summarized separately at the same grain. The resulting station hour tables can be joined only after their key uniqueness and coverage are checked. Station information adds current location and capacity for an initial screen. Weather values can be joined by local hour and the station to weather cell lookup, after checking coverage. MTA alert updates are collected, but their text must be converted into validated event intervals before they can be used as a disruption indicator. Stop coordinates support a stated distance rule once affected stops and event intervals are known.

A join will be checked for unintended row multiplication, unmatched station identifiers, and timestamps outside source coverage. The alert's publication time will not be treated as the disruption start time unless the source confirms that interpretation. Any method used to extract service times from alert text will be validated against known events.

## 2.5 Disruption exposure and analysis ready dataset

The confirmed closure windows describe when service was reduced, but the model also needs to know which stations are exposed. Two steps turn the windows into model inputs. First, the affected segment is taken from the MTA notices: no G trains run between Court Sq and Bedford-Nostrand Avs, which covers eight G stops (Court Sq, 21 St, Greenpoint Av, Nassau Av, Metropolitan Av, Broadway, Flushing Av and Myrtle-Willoughby Avs). The segment is checked against hourly ridership: a G station complex is flagged by ridership when its median ridership inside the closure windows falls below 35 percent of its typical level for that weekday and hour. Second, each Greenpoint station is assigned the distance in miles to its nearest affected stop, and a station is exposed when that distance is at most a quarter mile, which is the exposure rule in the project proposal.


In [ ]:
segment_complexes = g_ridership[["station_complex_id", "station_complex"]].drop_duplicates().copy()
segment_complexes["station_complex_id"] = segment_complexes["station_complex_id"].astype(str)

typical_ridership = ridership_wide.groupby(hour_keys).transform("median").replace(0, np.nan)
all_ratio = ridership_wide / typical_ridership
in_window_ratio = all_ratio.loc[all_ratio.index.isin(closure_hours)].median()
outside_ratio = all_ratio.loc[~all_ratio.index.isin(closure_hours)].median()
segment_table = segment_complexes.merge(
    pd.DataFrame({
        "station_complex": in_window_ratio.index,
        "ratio_inside_windows": in_window_ratio.to_numpy(),
        "ratio_outside_windows": outside_ratio.to_numpy(),
    }),
    on="station_complex",
)
segment_table["affected"] = segment_table["ratio_inside_windows"] < 0.35
segment_table = segment_table.sort_values("ratio_inside_windows").reset_index(drop=True)
affected_complex_ids = set(segment_table.loc[segment_table["affected"], "station_complex_id"])

# The MTA notices name the segment: no G trains between Court Sq (G22) and Bedford-Nostrand Avs (G33)
g_line_stops = mta_g_stops[mta_g_stops["gtfs_stop_id"].str.startswith("G")].copy()
g_line_stops["stop_number"] = g_line_stops["gtfs_stop_id"].str[1:].astype(int)
segment_stop_ids = g_line_stops.loc[g_line_stops["stop_number"].between(22, 32), "gtfs_stop_id"]
notice_complex_ids = set(
    mta_g_stops.loc[mta_g_stops["gtfs_stop_id"].isin(segment_stop_ids), "complex_id"].astype(str)
)
segment_table["named_in_notice"] = segment_table["station_complex_id"].isin(notice_complex_ids)
affected_stops = mta_g_stops[mta_g_stops["gtfs_stop_id"].isin(segment_stop_ids)].copy()
mta_g_stops["affected_segment"] = mta_g_stops["gtfs_stop_id"].isin(segment_stop_ids)

# Nearest affected stop, quarter mile exposure flag and distance band per Greenpoint station (built by citibike-prepare)
station_exposure = read_table("greenpoint_station_exposure").sort_values("station_id").reset_index(drop=True)
station_exposure["distance_band"] = tf.as_distance_band(station_exposure["distance_band"])

print(f"Confirmed closure hours: {len(closure_hours):,}")
print(f"G complexes with a ridership collapse inside the confirmed windows: {len(affected_complex_ids)} of {len(segment_table)}\nG complexes named in the notices: {len(notice_complex_ids)}; ridership flagged complexes also named in the notices: {len(affected_complex_ids & notice_complex_ids)} of {len(affected_complex_ids)}")
display(segment_table.assign(
    ratio_inside_windows=segment_table["ratio_inside_windows"].round(3),
    ratio_outside_windows=segment_table["ratio_outside_windows"].round(3),
))
print("Affected segment stops: " + ", ".join(affected_stops.sort_values("gtfs_stop_id")["stop_name"]))
print(f"Stations within 0.25 mile of an affected stop: {int(station_exposure['exposed_station'].sum())} of {len(station_exposure)}")
display(station_exposure["distance_band"].value_counts().sort_index().rename("stations").to_frame())

**Interpretation of the exposure definition.** Six of the 21 G complexes (Greenpoint Av, Nassau Av, Broadway, Myrtle-Willoughby Avs, 21 St and Flushing Av) keep only 1 to 5 percent of their usual entries inside the confirmed windows, while the two transfer complexes stay near normal. All six are also named in the MTA notices, so the ridership check agrees with the notices. The notices name eight stops. The two extra stops, Court Sq and Metropolitan Av, are transfer complexes (ratios 1.09 and 0.86) whose ridership is carried by other lines, so ridership alone cannot flag them. The notice based segment is used for exposure. Of the 43 Citi Bike stations, 14 lie within 0.25 mile of an affected stop, 21 between 0.25 and 0.5 mile, and 8 beyond 0.5 mile. The 14 stations form the exposed group. A limit is that distance to a stop is a proxy for being served by that stop, and riders may walk farther than a quarter mile.

### Station hour panel

The analysis table has one row per station and clock hour from August 1 to September 30, 2026. Trip counts are observed for the whole period, so a station hour with no recorded trip is a true zero. Station status begins on August 21, so earlier rows keep missing availability values instead of zeros, and the `status_observed` flag marks the rows that have them. Weather is joined by hour, exposure and closure flags are added, and calendar and lag features are created. The table is checked for duplicate keys and the expected row count, then saved as the `analysis_station_hour_panel` table by the `citibike-prepare` job; the cell below loads it and repeats the checks.

In [ ]:
PANEL_START, PANEL_END, TRIP_COVERAGE_START = study.PANEL_START, study.PANEL_END, study.TRIP_COVERAGE_START
panel_hours = pd.date_range(PANEL_START, PANEL_END, freq="h")

# Built by tf.greenpoint_panel in the citibike-prepare job
panel = read_table("analysis_station_hour_panel").sort_values(["station_id", "hour"]).reset_index(drop=True)
panel["distance_band"] = tf.as_distance_band(panel["distance_band"])

if panel.duplicated(["station_id", "hour"]).any():
    raise ValueError("The analysis panel contains duplicate station hour keys.")
expected_rows = len(station_exposure) * len(panel_hours)
if len(panel) != expected_rows:
    raise ValueError(f"Panel has {len(panel):,} rows, expected {expected_rows:,}.")

panel_checks = pd.DataFrame([
    ("Stations", panel["station_id"].nunique()),
    ("Hours", len(panel_hours)),
    ("Station hour rows", len(panel)),
    ("Rows with observed trip counts (August and September)", int(panel["trip_observed"].sum())),
    ("Rows with status snapshots", int(panel["status_observed"].sum())),
    ("Rows missing weather", int(panel["temperature_2m"].isna().sum())),
    ("Closure hours (all stations)", int(panel["g_closure_hour"].sum())),
    ("Closure station hours at exposed stations", int(panel["exposed_closure_hour"].sum())),
    ("Closure station hours with observed trips", int((panel["g_closure_hour"] & panel["trip_observed"]).sum())),
], columns=["Check", "Result"])
missing_by_column = panel.isna().sum().loc[lambda counts: counts.gt(0)].rename("missing values").to_frame()
missing_by_column["share"] = (missing_by_column["missing values"] / len(panel)).round(3)

print(f"Analysis panel loaded: {panel.shape[0]:,} rows and {panel.shape[1]} columns")
display(panel_checks)
display(missing_by_column)

**Interpretation of the analysis panel.** The panel has one row per station and hour: 43 stations over 1,464 hours gives 62,952 rows, and every row has observed trip counts and weather. Status snapshots exist for 41,512 rows (66 percent). The 21,440 rows without status are all hours before August 21 and share the same missingness pattern, so availability features can only be modeled on the later period. The lag features lose 1,032 rows (24 hour lag) and 7,224 rows (168 hour lag) at the start of the series. The 220 confirmed closure hours give 9,460 closure station hours, of which 3,080 are at the 14 exposed stations. This panel rests on four closure weekends, and the extended panel in Section 2.6 widens that to 13.

## 2.6 Extended datasets for modeling

The analysis panel in Section 2.5 covers only the Greenpoint stations in August and September 2026. The project proposal needs more than that: a shutdown table, demand for corridor and control stations, a baseline built from recent normal weeks, a demand proxy from station status, and a 2024 shutdown check. The `citibike-prepare` job builds those tables from the trip months listed in Section 2.1 and saves each one as a Delta table; the cells below load them.

| Table | Variable | Delta table | Purpose |
|---|---|---|---|
| Shutdown table | `mta_subway_shutdowns` | `mta_subway_shutdowns` | One row per shutdown event and affected G stop, with the Citi Bike stations within 0.25 mile |
| Station sets | `model_stations` | `model_station_sets` | Corridor stations and a random sample of control stations |
| Hourly trips | `trip_hourly` | `trips` (counted on load) | Departures and arrivals per station hour, May to September 2024 and January to September 2026 |
| Extended panel | `panel_extended` | `analysis_station_hour_panel_extended` | Demand, weather, calendar, shutdown flags, baseline, profile, split and status proxy |
| Accuracy check | `baseline_accuracy` | `baseline_accuracy_by_month` | Error of the baseline by month and station group |
| Effect screening | `event_effect_table` | `shutdown_effect_screening` | Demand at exposed versus control stations during shutdowns and placebo weekends |

In [ ]:
EXTENDED_MONTHS = tuple(month.replace("-", "") for month in study.EXTENDED_MONTHS)
CONTROL_MIN_MILES = study.CONTROL_MIN_MILES
EXPOSURE_MILES = study.EXPOSURE_MILES

# Corridor stations and the seeded control sample (tf.station_sets in the citibike-prepare job, seed 622)
model_stations = read_table("model_station_sets").sort_values(["station_group", "short_name"]).reset_index(drop=True)
g_only_stops = mta_g_stops.loc[
    mta_g_stops["gtfs_stop_id"].str.startswith("G"), ["gtfs_stop_id", "stop_name", "gtfs_latitude", "gtfs_longitude"]
].copy()
model_pairs = model_stations[["short_name", "lat", "lon"]].merge(g_only_stops, how="cross")
model_pairs["distance_miles"] = tf.haversine_miles(
    model_pairs["lat"], model_pairs["lon"], model_pairs["gtfs_latitude"], model_pairs["gtfs_longitude"]
)

# Station pool behind the sample, for the counts below
stations_table = read_table("stations")
all_stations = stations_table[stations_table["capacity"].gt(0) & stations_table["short_name"].notna()].drop_duplicates("short_name")
in_corridor = all_stations["station_group"].eq("corridor")
control_pool = all_stations[~in_corridor & all_stations["miles_to_nearest_g_stop"].gt(CONTROL_MIN_MILES)]

# One row per shutdown event: thirteen 2026 closure weekends and the three 2024 phases (dates from the MTA notices)
shutdown_events = tf.shutdown_events(g_confirmed_pre_august, g_confirmed_windows, segment_stop_ids)
mta_subway_shutdowns = read_table("mta_subway_shutdowns")

print(f"Citi Bike stations with a trip station code and capacity: {len(all_stations):,}")
print(f"Corridor stations: {int((model_stations['station_group'] == 'corridor').sum())}; "
      f"control stations sampled: {int((model_stations['station_group'] == 'control').sum())} "
      f"from a pool of {len(control_pool):,} stations more than {CONTROL_MIN_MILES:.0f} miles from any G stop")
print(f"Shutdown events: {len(shutdown_events)}; shutdown table rows (event by stop): {len(mta_subway_shutdowns)}")
display(shutdown_events.assign(stops=shutdown_events["stop_ids"].str.len())
        .groupby("event_type").agg(events=("event_id", "size"), stops=("stops", "max"),
                                   first_start=("start_local", "min"), last_end=("end_local", "max")))
display(mta_subway_shutdowns.groupby("event_type")["citi_bike_stations_within_quarter_mile"].agg(["min", "max"]))

**Station sets and shutdown events.** A station is in the corridor group when it is in the Greenpoint screening area or within 0.25 mile of any G stop. That gives 94 corridor stations. The control group is a random sample of 60 stations drawn with seed 622 from the 1,048 stations that are more than 3 miles from every G stop. These stations are unaffected by G service, so they absorb citywide shifts such as weather, season and holidays.

The shutdown table has 16 events and 118 event by stop rows: the 13 confirmed 2026 weekends (Section 2.2.1) and the three phases of the 2024 shutdown. For 2024 the phases are Court Sq to Nassau Av (June 28 to July 5), Court Sq to Bedford-Nostrand Avs (July 5 to August 12) and Bedford-Nostrand Avs to Church Av (August 12 to September 3), taken from the MTA notice text. Stops closed in each phase are Court Sq, 21 St and Greenpoint Av for phase 1, the eight stops of the 2026 segment for phase 2, and Classon Av, Clinton-Washington Avs and Fulton St for phase 3. Only the G line stops are listed, because the shared F stops keep other service. The 2024 windows use whole days because the exact start and end times were not available, and are marked `times_assumed`. Eight of the nine earlier 2026 weekends also carry assumed times. Citi Bike stations are counted within 0.25 mile of each stop, the same exposure radius used in Section 2.5.

In [ ]:
# Departures and arrivals per model station and hour from the 14 trip files, counted in Databricks from the cleaned
# trips table (trips with a missing time or a nonpositive duration were dropped by the citibike-prepare job)
trip_hourly = data_access.hourly_station_flows(study.EXTENDED_MONTHS, model_stations["short_name"], spark=spark, config=CONFIG)
trip_hourly["month"] = trip_hourly["hour"].dt.to_period("M").astype(str)
coverage = trip_hourly.groupby("month").agg(
    stations=("short_name", "nunique"), station_hours=("short_name", "size"),
    departures=("departures", "sum"), arrivals=("arrivals", "sum"),
)
print(f"Hourly trip table for the model stations: {len(trip_hourly):,} station hours")
display(coverage)

**Hourly trips.** One pass over 58.4 million trip records (14 months) kept the trips that start or end at the 154 model stations (one control station, JC124, has no trips, so 153 stations remain) and counted departures and arrivals per station hour. Trips with a missing time or a nonpositive duration were dropped, as in Section 2.1. The result has 917,942 station hours. The first table shows that each month has between 126 and 149 active stations, and that January and February 2026 have lower volumes than summer months, which is the expected seasonal pattern. The few records dated April 2024 and December 2025 are trips whose end time falls in the next month file, and they lie outside the panel periods. The counts are computed in Databricks from the `trips` table, so only the hourly totals are transferred.

In [ ]:
# Built by tf.build_extended_panel in the citibike-prepare job: outcomes, weather, calendar, shutdown flags, split,
# status, station activity, baseline, profile and status proxies
panel_extended = (
    read_table("analysis_station_hour_panel_extended")
    .sort_values(["short_name", "period", "hour"]).reset_index(drop=True)
)
if panel_extended.duplicated(["short_name", "hour"]).any():
    raise ValueError("The extended panel contains duplicate station hour keys.")
extended_checks = pd.DataFrame([
    ("Stations", panel_extended["short_name"].nunique()),
    ("Station hour rows", len(panel_extended)),
    ("Rows in 2024 / 2026", f"{int(panel_extended['period'].eq('2024').sum()):,} / {int(panel_extended['period'].eq('2026').sum()):,}"),
    ("Rows missing weather", int(panel_extended["temperature_2m"].isna().sum())),
    ("Closure station hours", int(panel_extended["g_closure_hour"].sum())),
    ("Closure station hours at exposed stations", int(panel_extended["exposed_closure_hour"].sum())),
    ("Placebo weekend station hours", int(panel_extended["placebo_weekend_hour"].sum())),
    ("Holiday station hours", int(panel_extended["is_holiday"].sum())),
    ("Rows with status snapshots", int(panel_extended["status_observed"].sum())),
    ("Rows outside a station active span (flagged, not zero demand)", int((~panel_extended["station_active"]).sum())),
], columns=["Check", "Result"])
display(extended_checks)
display(panel_extended.groupby(["period", "split", "station_group"]).agg(
    stations=("short_name", "nunique"), rows=("short_name", "size"), mean_total_trips=("total_trips", "mean")).round(3))

**Extended panel.** The panel has one row per station and hour for two periods: May 1 to September 30, 2024 and January 1 to September 30, 2026. A station enters a period only if it has at least one trip in that period, because stations that did not exist yet would otherwise appear as zero demand. Missing station hours are true zeros, since the trip files record every trip.

The panel has 1,485,072 rows, 153 stations, no missing weather values, 325,045 closure station hours (84,346 at exposed stations), 212,004 placebo weekend station hours and 38,328 holiday station hours. Placebo weekends use the same Friday 22:00 to Monday 04:00 window on 2026 weekends with no known closure. They let later models and checks ask whether an apparent closure effect also shows up when nothing happened. Three of the 16 closure weekends in the proposal were not found in MTA notices, so a few placebo hours may be real closures. This would bias the placebo comparison toward the closure effect, so it is a limit to report.

The `split` column is chronological. January to June 2026 is `train`, July 2026 is `validation`, August and September 2026 is `test`, and 2024 is `validation_2024`, an external check of the shutdown effect. The `station_active` flag marks the 72,192 station hours (4.9 percent) that fall before a station's first recorded trip or after its last one in a period, which are stations not yet open or already removed. Those hours are zero filled in the panel but are not zero demand, so the baseline, the profile and Figures 6, 9 and 10 exclude them. Status columns are filled only for the 43 Greenpoint stations after August 21, 2026 (41,512 rows), because that is the only period with collected status.

In [ ]:
# Baseline: mean of the same station, weekday and hour in the previous four weeks (closure hours excluded, at least
# two weeks). Profile: mean per station, weekday and hour over the normal reference window. Both are columns of the
# extended panel; the accuracy table is built by tf.baseline_accuracy in the citibike-prepare job.
poisson_deviance = tf.poisson_deviance
baseline_accuracy = read_table("baseline_accuracy_by_month").sort_values(["month", "station_group"]).reset_index(drop=True)
print(f"Station hours with a baseline: {int(panel_extended['baseline_total_trips'].notna().sum()):,} of {len(panel_extended):,}")
print(f"Station hours with a profile: {int(panel_extended['profile_total_trips'].notna().sum()):,}")
display(baseline_accuracy.round(3))

**Baseline and normal profile.** The baseline for a station hour is the mean demand for the same station, day of week and hour in the previous four weeks, excluding closure hours, and it requires at least two usable weeks. It has no demand lags beyond the weekly pattern, so it can be used as a predictor without leaking the current hour. The profile is the mean demand per station, day of week and hour over normal hours in a reference window (February to July 2026, or May 1 to June 27, 2024). Hours before a station opened or after it was removed are excluded from the baseline and the profile (see the `station_active` flag). The profile is built from the same period it describes, so it is a descriptive normal level and not an out of sample forecast. 1,095,452 station hours have a baseline. The missing ones are mostly in 2024 (253,793) because the shutdown covers most of July and August, so no normal weeks remain in the four week window.

The accuracy table reports mean absolute error (MAE) and Poisson deviance by month, excluding closure and holiday hours. Corridor stations have larger errors because their volumes are higher: the average MAE is about 3.2 trips per station hour for the corridor and 1.1 for control stations, against mean demand of 2 to 9 and 0.4 to 2 trips. The Poisson deviance is about 3.3 for the corridor and 1.6 for control. August 2026 corridor deviance is higher (4.9) because one station had no trips in the prior weeks, so its baseline is zero. Deviance uses a floor of 0.1 on the prediction to avoid an infinite penalty. These numbers set the accuracy that the models in Section 4 must beat.

In [ ]:
# Status derived demand proxy: the change in mean bikes available between consecutive hours approximates net flow.
# The proxy columns are already in the extended panel; this recomputes them with the two hour flow averages to check them.
status_panel = tf.status_proxy_frame(panel_extended)
proxy_check = tf.proxy_check_table(status_panel)
display(proxy_check)

**Status derived demand proxy.** Status data records only the number of bikes and docks available. A drop in the hourly mean of bikes available suggests departures and a rise suggests arrivals, but rebalancing by Citi Bike staff moves bikes without trips. The check compares the hourly change in bikes available with the average net flow (arrivals minus departures) of the same and previous hour, for the 41,469 station hours with consecutive status. The correlation with net flow is 0.87 and the signs agree in 87 percent of nonzero hours, so the proxy tracks the direction and size of net flow well. The mean absolute error is 1.03 bikes per hour. The correlation of the departure proxy with departures alone is only 0.42, because hourly means mix arrivals and departures in the same hour. The proxy is therefore useful for net flow and for stations without trip data, but not as a replacement for trip counts.

In [ ]:
import matplotlib.pyplot as plt

# Figure 9: 2024 shutdown check. Daily trips per station relative to the pre shutdown mean, exposed versus control.
p24 = panel_extended[panel_extended["period"].eq("2024") & panel_extended["station_active"]].copy()
p24["group"] = np.where(p24["exposed_station_2024"], "Within 0.25 mile of a 2024 closed stop",
                        np.where(p24["station_group"].eq("control"), "Control stations (more than 3 miles away)", "Other corridor stations"))
p24 = p24[~p24["group"].eq("Other corridor stations")]
daily = p24.groupby(["group", p24["hour"].dt.normalize()])["total_trips"].sum().rename("trips").reset_index()
stations_per_group = p24.groupby("group")["short_name"].nunique()
daily["per_station"] = daily["trips"] / daily["group"].map(stations_per_group)
pre_mean = daily[daily["hour"] < "2024-06-28"].groupby("group")["per_station"].mean()
daily["index"] = daily["per_station"] / daily["group"].map(pre_mean)
daily["smooth"] = daily.groupby("group")["index"].transform(lambda values: values.rolling(7, center=True, min_periods=4).mean())
fig, axis = plt.subplots(figsize=(12, 4.8))
for label, color in (("Within 0.25 mile of a 2024 closed stop", "tab:red"), ("Control stations (more than 3 miles away)", "tab:blue")):
    part = daily[daily["group"].eq(label)]
    axis.plot(part["hour"], part["smooth"], color=color, label=label)
for label, start, end, shade in (("Phase 1", "2024-06-28", "2024-07-05", "0.85"), ("Phase 2", "2024-07-05", "2024-08-12", "0.92"), ("Phase 3", "2024-08-12", "2024-09-03", "0.85")):
    axis.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=shade, zorder=0)
    axis.text(pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2, axis.get_ylim()[1], label, ha="center", va="top", fontsize=8)
axis.axhline(1, color="black", linewidth=0.8, linestyle=":")
axis.set_xlabel("Date (2024)")
axis.set_ylabel("Trips per station relative to May 1 to June 27 mean")
axis.set_title("Figure 9. Citi Bike trips around the 2024 G shutdown (7 day centered mean)")
axis.legend(fontsize=8, loc="lower left")
axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# Figure 10: actual to normal profile ratio at exposed versus control stations, by event type
# Actual to profile demand at exposed and control stations per event type (tf.shutdown_effect_screening; the
# citibike-prepare job saves the same table as shutdown_effect_screening)
event_effect_table = tf.shutdown_effect_screening(panel_extended)
wide = event_effect_table.pivot(index="event", columns="group", values="actual_to_profile")
wide["exposed_to_control"] = wide["Exposed stations"] / wide["Control stations"]
order = ["2026 placebo weekends", "2026 closure weekends", "2024 phase 1", "2024 phase 2", "2024 phase 3"]
wide = wide.loc[order]
fig, axis = plt.subplots(figsize=(9, 4.5))
bars = axis.bar(wide.index, wide["exposed_to_control"], color=["0.6", "tab:red", "tab:orange", "tab:orange", "tab:orange"])
axis.axhline(1, color="black", linewidth=0.8, linestyle=":")
axis.bar_label(bars, fmt="%.2f", fontsize=8)
axis.set_ylabel("Exposed to control ratio of actual to normal trips")
axis.set_title("Figure 10. Relative demand at exposed stations during shutdowns and placebo weekends")
axis.tick_params(axis="x", labelrotation=15)
axis.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()
display(wide.round(3))

**Figure 9 and Figure 10 explanation.** Figure 9 shows trips per station relative to the May 1 to June 27, 2024 mean, as a seven day centered average, for stations within 0.25 mile of a closed 2024 stop and for control stations. Both lines rise into summer, which is seasonal growth and not a shutdown effect. The line for exposed stations sits above the control line through the three shutdown phases (the shaded bands), which means demand near the closed stops grew faster than demand elsewhere.

Figure 10 turns that comparison into one number per event type: the ratio of actual trips to the normal profile at exposed stations divided by the same ratio at control stations. A value of 1 means no relative change. Placebo weekends give 1.01, so the method finds no effect when nothing happened. Confirmed 2026 closure weekends give 1.17. The three 2024 phases give 1.16, 1.12 and 1.04. The 2024 phase 2 value of about 12 percent is close to the nearly 14 percent increase reported in the proposal. These values are descriptive screening results: they support the main hypothesis that demand rises near closed stops, but they do not control for weather, hour of day or station size. The models in Section 4 will do that. Phase 1 and phase 3 have few exposed station hours (2,923 and 6,206), so they are less certain than phase 2 (39,547).

# 3. Exploration: exploratory data analysis and feature engineering

Exploratory analysis will describe the data before any predictive or causal model is fit. Tables and plots will include the sample size, time span, and aggregation level needed to interpret them. Every plot in this section is generated from the local data, states its sample and time span, and is followed by an interpretation.

## 3.1 Coverage, data quality, and study period

The data quality table in Section 2 reports source rows, excluded endpoint records, station identifiers that do not match the latest station file, and the overlap between trip and status hours. The local trip records used for the area figures cover May to September 2024 and January to September 2026 (hourly counts built in Section 2.6). Historical station status covers August 21 through October 1, 2026. Weather observations cover the same trip months, and the subway stop lookup provides coordinates for G train stops. MTA alert updates currently end on August 11, 2026, before the status sample, and the live alert snapshot only lists G work from October 19, 2026. Some rides in each trip archive start or end just outside the month in its file name, so the timestamp of each measured event defines the analysis period. Closure intervals are taken from MTA planned service notices for 13 weekends in 2026 (Section 2.2.1) and from the 2024 shutdown phases (Section 2.6). Only the three weekends from August 21 onward overlap station status.

Trip records from August 21 to September 30, 2026 overlap with station status in time. Trips in the other months do not have matching historical station status in the local files, and status continues beyond the trip period. The two sources are therefore analyzed separately except where station and hour keys match. This prevents a missing record in one source from being interpreted as a measured zero in another.

In [ ]:
import matplotlib.pyplot as plt

# Area totals for the 43 Greenpoint stations over all extended trip months (2024 and 2026)
area_codes = set(greenpoint_stations["trip_station_id"].dropna())
area_hourly_all = (
    trip_hourly[trip_hourly["short_name"].isin(area_codes)]
    .groupby("hour")[["departures", "arrivals"]].sum()
)
MONTHS_FOR_EXPLORATION = tuple(
    [f"2024-{month:02d}" for month in range(5, 10)] + [f"2026-{month:02d}" for month in range(1, 10)]
)
monthly_hourly = {}
monthly_daily = {}
for month in MONTHS_FOR_EXPLORATION:
    month_start = pd.Timestamp(f"{month}-01")
    month_end = month_start + pd.offsets.MonthBegin(1)
    hour_index = pd.date_range(month_start, month_end, freq="h", inclusive="left")
    hourly = area_hourly_all.reindex(hour_index, fill_value=0)
    monthly_hourly[month] = hourly
    monthly_daily[month] = hourly.resample("D").sum()

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
for axis, year in zip(axes, ("2024", "2026")):
    daily = pd.concat([monthly_daily[m] for m in MONTHS_FOR_EXPLORATION if m.startswith(year)])
    axis.plot(daily.index, daily["departures"], label="Departures", color="#247ba0", linewidth=1)
    axis.plot(daily.index, daily["arrivals"], label="Arrivals", color="#f28e2b", linewidth=1)
    if year == "2024":
        for label, start, end, shade in (("Phase 1", "2024-06-28", "2024-07-05", "0.8"),
                                         ("Phase 2", "2024-07-05", "2024-08-12", "0.9"),
                                         ("Phase 3", "2024-08-12", "2024-09-03", "0.8")):
            axis.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=shade, zorder=0)
            axis.text(pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2,
                      daily[["departures", "arrivals"]].max().max() * 1.02, label, ha="center", fontsize=8)
    else:
        for window in pd.concat([g_confirmed_pre_august[["start_local", "end_local"]],
                                 g_confirmed_windows[["start_local", "end_local"]]]).itertuples():
            axis.axvspan(window.start_local, window.end_local, color="tab:red", alpha=0.25, zorder=0)
    axis.set_title(f"{year}: daily trips in the Greenpoint screening area (shaded: G service reductions)")
    axis.set_ylabel("Trips per day")
    axis.grid(alpha=0.3)
    axis.legend(frameon=False, loc="upper left")
axes[-1].set_xlabel("Date")
fig.suptitle("Daily Citi Bike trips in Greenpoint", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

daily_totals = pd.DataFrame(
    {month: monthly_daily[month][["departures", "arrivals"]].sum() for month in MONTHS_FOR_EXPLORATION}
).T
daily_totals["calendar days"] = [len(monthly_daily[month]) for month in MONTHS_FOR_EXPLORATION]
daily_totals["mean departures per day"] = daily_totals["departures"] / daily_totals["calendar days"]
daily_totals["mean arrivals per day"] = daily_totals["arrivals"] / daily_totals["calendar days"]
display(daily_totals.round(1))

**Figure 1. Daily Citi Bike departures and arrivals in the Greenpoint screening area, May to September 2024 and January to September 2026.** The top panel shows 2024 and the bottom panel shows 2026, with counts assigned by the recorded trip start and end times. Gray bands in the top panel mark the three phases of the 2024 G shutdown. Red bands in the bottom panel mark the 13 confirmed 2026 weekend closures. The months between October 2024 and December 2025 are not part of the study data, so they are not shown.

**Interpretation.** Demand follows a strong seasonal cycle. Mean departures per day rise from 1,269 in February 2026, the lowest month, to 5,268 in August 2026, the highest. Within 2024, mean daily departures grew from 3,258 in May to 4,583 in July and stayed near 4,300 to 4,600 through September. Arrivals track departures closely in every month (the two lines almost overlap), which is expected for a set of stations where bikes that leave are returned nearby, so the net flow of the area is small. The same months are higher in 2026 than in 2024 (June 5,253 against 4,327 departures per day, and September 5,035 against 4,560), which is consistent with system growth, with more stations (40 active in 2024, 43 in August and September 2026) and with different weather. The repeated waves are the weekly cycle of weekdays and weekends. The closure bands run from Friday night to Monday morning, so they sit on the weekend part of that cycle, and no lasting change in the level of the series is visible after them. Several days in late January 2026 have close to zero trips, and 2026 has a clear dip at the end of September. The causes were not investigated here, and such days will matter for the weather controls in later models. Area totals hide differences between stations, so these plots cannot show whether stations near a closed stop behaved differently. That comparison is made at the station level in Section 2.6 and in Section 3.6. The station count also changes between months (for example 36 active stations in July 2026), so month to month comparisons mix demand changes with availability of stations.

## 3.2 Daily and hourly demand patterns

Figure 1 summarizes daily departures and arrivals in the Greenpoint screening area. Figure 2 compares average area wide activity by local hour and day type. Each month is shown separately because seasonal conditions differ. These are descriptive summaries across selected stations, not individual rider behavior.

The daily figure above shows total volume by date. The next figure examines how this activity is distributed across the hours of a typical weekday and weekend day.

## 3.3 Hourly demand profiles

Hourly profiles use the complete calendar grid for each of the 14 months, with unobserved station hour counts represented as zero only inside the local area wide trip totals. The next subsection examines spatial differences and the separate station inventory records.

**Figure 2. Average hourly trips by day type.** The left panel shows weekdays and the right panel shows weekends. Each line is one month from May to September 2024 (dashed) and January to September 2026 (solid), colored from the earliest to the latest month. The lines show mean total departures plus arrivals per clock hour across the Greenpoint screening area. Hours without a recorded area wide trip are included as zero within each calendar month.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
hourly_profiles = {}
peak_rows = []
colors = dict(zip(MONTHS_FOR_EXPLORATION, plt.cm.viridis(np.linspace(0, 0.95, len(MONTHS_FOR_EXPLORATION)))))

for month in MONTHS_FOR_EXPLORATION:
    hourly = monthly_hourly[month].copy()
    hourly["day_type"] = hourly.index.dayofweek.map(lambda day: "Weekend" if day >= 5 else "Weekday")
    hourly["hour_of_day"] = hourly.index.hour
    hourly["trips"] = hourly["departures"] + hourly["arrivals"]
    profile = hourly.groupby(["day_type", "hour_of_day"], observed=True)["trips"].mean().unstack("day_type")
    hourly_profiles[month] = profile
    for axis, day_type in zip(axes, ("Weekday", "Weekend")):
        axis.plot(profile.index, profile[day_type], color=colors[month], linewidth=1.4,
                  linestyle="--" if month.startswith("2024") else "-", label=month)
        peak_hour = profile[day_type].idxmax()
        peak_rows.append({"month": month, "day type": day_type, "peak local hour": int(peak_hour),
                          "mean trips at peak": round(float(profile.loc[peak_hour, day_type]), 1)})
for axis, day_type in zip(axes, ("Weekday", "Weekend")):
    axis.set_title(day_type)
    axis.set_xlabel("Local hour of day")
    axis.set_xticks(range(0, 24, 3))
    axis.grid(alpha=0.3)
axes[0].set_ylabel("Mean trips across the area per clock hour")
axes[1].legend(frameon=False, fontsize=7, ncol=2, title="Month (dashed: 2024)", loc="upper left")
fig.suptitle("Hourly Citi Bike activity by day type", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

display(pd.DataFrame(peak_rows).pivot(index="month", columns="day type", values=["peak local hour", "mean trips at peak"]))

**Figure 2 interpretation.** The weekday profile has the same shape in every month: demand is low overnight, rises through the morning, and reaches its daily peak at 18:00 in 13 of the 14 months (17:00 in February 2026). The mean combined trips at the weekday peak range from 264 in February 2026 to 1,135 in September 2026. A morning peak at 08:00 is also clear and reaches roughly 60 to 70 percent of the evening peak, which is the commuter pattern of a residential area. The weekend profile has a single broad midday and afternoon hump, with its peak between 12:00 and 17:00, and in summer months more trips after midnight, with a peak of 239 in January 2026 and 938 in August 2026. The dashed 2024 lines have the same weekday shape as the 2026 lines, so the daily rhythm is stable across years and the 2024 data can be used as an external check of the shutdown effect. The curves rise from winter to late summer, which is the seasonal pattern also seen in Figure 1. These area summaries show when trip activity was highest in this sample, not why it was high. The hourly means include hours with no area trips. They do not control for weather or for the number of active stations, and the table of peak hours and peak means by month is the numeric form of the figure.

## 3.4 Spatial demand, station availability, and disruption limits

Station identifiers are matched to the latest station information snapshot, and station by hour keys are checked before plotting. Figure 3 ranks Greenpoint stations by total departures plus arrivals from January to September 2026 and checks whether the ranking held in summer 2024. Figure 4 measures the share of observed status snapshots with no open docks. A station with no bikes is a separate condition from one with no open docks. Station status exists only for August 21 to September 30, 2026, so Figure 4 cannot be extended to earlier months. The confirmed closure windows are linked to Citi Bike stations with a quarter mile distance rule in Sections 3.5 and 3.6, where the disruption comparison is made. That comparison is descriptive and is not a causal estimate.

**Figure 3. Station volume and stability across years.** The left panel ranks the ten busiest Greenpoint stations by departures plus arrivals from January to September 2026, with departures and arrivals stacked. The right panel compares the mean daily trips of the same stations in June to September 2024 and 2026.

In [ ]:
# Station ranking over all 2026 months, and the stability of the busiest stations against summer 2024
name_by_code = greenpoint_stations.set_index("trip_station_id")["name"]
area_trips = trip_hourly[trip_hourly["short_name"].isin(name_by_code.index)].copy()
area_trips["name"] = area_trips["short_name"].map(name_by_code)
trips_2026 = area_trips[area_trips["hour"].between("2026-01-01", "2026-09-30 23:00")]
ranking_2026 = trips_2026.groupby("name")[["departures", "arrivals"]].sum()
ranking_2026["total trips"] = ranking_2026["departures"] + ranking_2026["arrivals"]
top_stations = ranking_2026.nlargest(10, "total trips").sort_values("total trips")

def summer_daily_mean(frame, start, end):
    window = frame[frame["hour"].between(start, end + " 23:00")]
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    return (window.groupby("name")[["departures", "arrivals"]].sum().sum(axis=1) / days)
summer_2024 = summer_daily_mean(area_trips, "2024-06-01", "2024-09-30").reindex(top_stations.index).fillna(0)
summer_2026 = summer_daily_mean(area_trips, "2026-06-01", "2026-09-30").reindex(top_stations.index).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={"width_ratios": [1.2, 1]})
positions = np.arange(len(top_stations))
axes[0].barh(positions, top_stations["departures"], label="Departures", color="#247ba0")
axes[0].barh(positions, top_stations["arrivals"], left=top_stations["departures"], label="Arrivals", color="#f28e2b")
axes[0].set_yticks(positions, top_stations.index)
axes[0].set_xlabel("Trips, January to September 2026")
axes[0].set_title("Ten busiest stations, 2026")
axes[0].legend(frameon=False)
height = 0.38
axes[1].barh(positions - height / 2, summer_2024, height, label="June to September 2024", color="0.6")
axes[1].barh(positions + height / 2, summer_2026, height, label="June to September 2026", color="#247ba0")
axes[1].set_yticks(positions, [""] * len(positions))
axes[1].set_xlabel("Mean trips per day (departures plus arrivals)")
axes[1].set_title("The same stations in summer 2024 and summer 2026")
axes[1].legend(frameon=False)
for axis in axes:
    axis.grid(alpha=0.3, axis="x")
fig.suptitle("Busiest Greenpoint stations and their stability across years", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

top_table = top_stations.sort_values("total trips", ascending=False).copy()
top_table["share of area trips"] = top_table["total trips"] / ranking_2026["total trips"].sum()
top_table["mean daily trips, summer 2024"] = summer_2024
top_table["mean daily trips, summer 2026"] = summer_2026
display(top_table.round(3))
rank_2024 = summer_daily_mean(area_trips, "2024-06-01", "2024-09-30")
rank_2026 = summer_daily_mean(area_trips, "2026-06-01", "2026-09-30")
common = rank_2024.index.intersection(rank_2026.index)
print(f"Rank correlation of station volume, summer 2024 and summer 2026 ({len(common)} stations active in both): "
      f"{rank_2024[common].rank().corr(rank_2026[common].rank()):.3f}")
print(f"Top ten stations in 2026 that were also in the top ten in summer 2024: "
      f"{len(set(top_stations.index) & set(rank_2024.nlargest(10).index))} of 10")
print(f"Share of all area trips at the ten busiest stations: {top_stations['total trips'].sum() / ranking_2026['total trips'].sum():.1%}")



**Figure 3 interpretation.** Franklin St & Dupont St is the busiest station with 113,269 trips (56,872 departures and 56,397 arrivals), followed by Bedford Ave & Nassau Ave with 92,319 and Kent Ave & N 7 St with 90,940. The ten stations carry 41.6 percent of all trips in the area, so demand is concentrated. Departures and arrivals are nearly equal at every station, which means these stations are not persistent sources or sinks over the full period. The right panel shows stability: the rank correlation of station volume between summer 2024 and summer 2026 is 0.894 across the 40 stations active in both, and seven of the ten leading stations were already in the top ten in 2024. Volume rose at most stations, most strongly at India St Pier (263 to 454 mean daily trips). The ranking therefore supports using station level baselines across both years, but it does not identify which stations were affected by a closure.

**Figure 4. Share of station status snapshots reporting no open docks.** The figure compares weekdays and weekends by local hour over the available station status period, August 21 through October 1, 2026. The denominator is the number of station status observations, not the number of stations.

In [ ]:
availability_profile = (
    greenpoint_status.groupby(["day_type", "hour_of_day"], observed=True)
    .agg(
        zero_dock_snapshots=("zero_dock_snapshots", "sum"),
        zero_bike_snapshots=("zero_bike_snapshots", "sum"),
        snapshots=("snapshot_count", "sum"),
    )
)
availability_profile["zero dock percent"] = (
    100 * availability_profile["zero_dock_snapshots"] / availability_profile["snapshots"]
)
availability_profile["zero bike percent"] = (
    100 * availability_profile["zero_bike_snapshots"] / availability_profile["snapshots"]
)

fig, axis = plt.subplots(figsize=(10, 5))
for day_type, color in (("Weekday", "#247ba0"), ("Weekend", "#f28e2b")):
    series = availability_profile.xs(day_type, level="day_type")["zero dock percent"]
    axis.plot(series.index, series, marker="o", markersize=3, label=day_type, color=color)
axis.set_xlabel("Local hour of day")
axis.set_ylabel("Snapshots with zero open docks (%)")
axis.set_xticks(range(0, 24, 2))
axis.set_title("Open dock availability by hour and day type", fontweight="bold")
axis.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Overall snapshot shares by day type")
display(availability_profile.groupby(level="day_type").apply(
    lambda group: pd.Series({
        "zero dock percent": 100 * group["zero_dock_snapshots"].sum() / group["snapshots"].sum(),
        "zero bike percent": 100 * group["zero_bike_snapshots"].sum() / group["snapshots"].sum(),
        "snapshots": group["snapshots"].sum(),
    })
).round(2))

**Figure 4 interpretation.** Across the collected snapshots, zero open docks were reported in 14.56% of weekday observations and 13.34% of weekend observations. Zero bikes were less common, at 1.64% and 0.71%, respectively. This suggests that the observed inventory constraint was more often a lack of return capacity than a lack of bikes, within this station screen and period. These percentages describe snapshots, not every moment between feed updates, and they do not show that a subway closure caused the inventory pattern.

## 3.5 Feature engineering

The preparation cell creates calendar features from the local event time and joins current station attributes using station identifiers. Candidate features for later modeling include:

1. **Hour of day.** The local hour from the analysis timestamp represents daily travel patterns. Confirm local time and daylight saving handling.
2. **Day of week.** The local weekday separates weekday and weekend travel. Confirm category ordering and holidays.
3. **Weekend indicator.** Whether the date falls on Saturday or Sunday provides a simple day type summary. Compare with a full weekday category.
4. **Station attributes.** Location and capacity represent differences in station setting and size. Check dated capacity and identifier matches.
5. **Weather measures.** Temperature and precipitation from the historical archive may account for travel conditions. The station to grid lookup follows the supplied Citi Bike pipeline reference. Check units, local time, spatial resolution, coverage, and missingness before modeling.
6. **Service disruption indicator.** Whether a verified event affects a station and hour identifies event timing and exposure. Use actual service times, not notice publication time alone.
7. **Distance to affected stop.** Geographic distance allows effects to vary with proximity. State the distance method and test more than one threshold.
8. **Recent demand summaries.** Past station activity represents local demand history. Use only information available at prediction time and prevent future data leakage.

The calendar features created for exploration are local hour, weekday, weekday or weekend, date, and calendar month. They describe regular time patterns and support the figures above. Weather, closure flags (`g_closure_hour`, `exposed_closure_hour`), distance to the nearest affected stop, lags and the weekly baseline are joined in the panels of Sections 2.5 and 2.6. Any later model feature must be available at forecast time. Station status measures describe reported inventory, not trip counts.

## 3.6 Exposure map and the effect of confirmed G reductions

Figure 5 maps the exposure definition for the 94 corridor stations. Figure 6 then compares exposed, other corridor and control stations during 2026 weekend hours with and without a confirmed G reduction, using the 13 confirmed weekends, dry hours only and excluding holidays. The comparison uses only hours of the day that occur in a closure window, so that the groups cover the same clock hours. The outcome is the ratio of actual trips to each station's normal profile for the same weekday and hour. The comparison is descriptive: it does not adjust for events or station differences.

In [ ]:
from matplotlib.patches import Ellipse

# Distance from every model station (corridor group) to the nearest stop of the affected 2026 segment
segment_pairs = model_pairs[model_pairs["gtfs_stop_id"].isin(segment_stop_ids)]
map_stations = model_stations.merge(
    segment_pairs.groupby("short_name")["distance_miles"].min().rename("segment_miles"),
    on="short_name", how="left",
)
map_stations = map_stations[map_stations["station_group"].eq("corridor")].copy()
map_stations["distance_band"] = pd.cut(
    map_stations["segment_miles"], [0, 0.25, 0.5, np.inf],
    labels=["Within 0.25 mile", "0.25 to 0.5 mile", "Beyond 0.5 mile"], include_lowest=True,
)
map_stations["in_greenpoint_screen"] = map_stations["short_name"].isin(greenpoint_stations["short_name"])

fig, ax = plt.subplots(figsize=(8.5, 8.5))
band_colors = {"Within 0.25 mile": "tab:red", "0.25 to 0.5 mile": "tab:orange", "Beyond 0.5 mile": "tab:blue"}
view = dict(lat=(40.690, 40.760), lon=(-73.990, -73.915))
visible = map_stations[map_stations["lat"].between(*view["lat"]) & map_stations["lon"].between(*view["lon"])]
for band, color in band_colors.items():
    subset = visible[visible["distance_band"] == band]
    ax.scatter(subset["lon"], subset["lat"], s=40, color=color, edgecolor="white",
               label=f"Citi Bike station, {band.lower()} ({len(subset)})", zorder=3)

near_area = mta_g_stops[mta_g_stops["gtfs_latitude"].between(*view["lat"]) & mta_g_stops["gtfs_longitude"].between(*view["lon"])]
affected_near = near_area[near_area["affected_segment"]]
unaffected_near = near_area[~near_area["affected_segment"]]
ax.scatter(affected_near["gtfs_longitude"], affected_near["gtfs_latitude"], marker="s", s=90, color="black", label="G stop in affected segment", zorder=4)
ax.scatter(unaffected_near["gtfs_longitude"], unaffected_near["gtfs_latitude"], marker="s", s=90, facecolor="white", edgecolor="black", label="G stop outside affected segment", zorder=4)
for stop in affected_near.itertuples():
    ax.annotate(stop.stop_name, (stop.gtfs_longitude, stop.gtfs_latitude), textcoords="offset points", xytext=(6, 5), fontsize=8)
    radius_lat = 0.402336 / 111.32
    radius_lon = 0.402336 / (111.32 * np.cos(np.radians(stop.gtfs_latitude)))
    ax.add_patch(Ellipse((stop.gtfs_longitude, stop.gtfs_latitude), 2 * radius_lon, 2 * radius_lat, fill=False, linestyle="--", color="gray", linewidth=0.8))

ax.set_xlim(*view["lon"])
ax.set_ylim(*view["lat"])
ax.set_aspect(1 / np.cos(np.radians(40.725)))
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Corridor Citi Bike stations by distance to the nearest affected G stop")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

print(f"Corridor stations in the map window: {len(visible)} of {len(map_stations)}")
print(f"Within 0.25 mile of the affected segment: {int(map_stations['distance_band'].eq('Within 0.25 mile').sum())} "
      f"({int((map_stations['distance_band'].eq('Within 0.25 mile') & map_stations['in_greenpoint_screen']).sum())} in the Greenpoint screen)")
display(map_stations.groupby("distance_band", observed=True).agg(
    stations=("short_name", "size"), mean_capacity=("capacity", "mean"),
    median_miles=("segment_miles", "median")).round(2))


**Interpretation of Figure 5.** The map shows the 77 of the 94 corridor model stations that fall inside the plotted window, colored by distance to the nearest stop of the affected G segment (Court Sq to Bedford-Nostrand Avs), with quarter mile circles around each stop. In the plotted window, 46 stations lie within 0.25 mile (median 0.18 mile), 23 lie between 0.25 and 0.5 mile and 8 lie beyond 0.5 mile. Across all 94 corridor stations the counts are 46, 24 and 24. Of the 46 exposed stations, 14 are inside the Greenpoint screening rectangle, so the extended station set adds Long Island City and Williamsburg stations to the exposed group. Mean dock capacity is similar across bands (29.2, 34.5 and 31.2). The exposed group is spatially concentrated, so a difference between exposed and other stations may also reflect neighborhood density and land use. The map shows geometry only and says nothing about travel behavior.

In [ ]:
# Weekend hours in 2026 with and without a confirmed G reduction, dry hours only, all model stations
weekend_all = panel_extended[
    panel_extended["period"].eq("2026") & panel_extended["station_active"] & panel_extended["is_weekend"] & ~panel_extended["is_holiday"]
    & panel_extended["hour_of_day"].between(6, 23) & ~panel_extended["precipitation_any"]
].copy()
closure_hour_set = sorted(weekend_all.loc[weekend_all["g_closure_hour"], "hour_of_day"].unique())
comparison = weekend_all[weekend_all["hour_of_day"].isin(closure_hour_set)].copy()
comparison["group"] = np.select(
    [comparison["exposed_station"], comparison["station_group"].eq("corridor")],
    ["Exposed (within 0.25 mile)", "Other corridor"], default="Control (more than 3 miles)",
)
# Ratio of actual trips to each station's normal profile for the same weekday and hour, summed over stations so large stations weigh more
comparison = comparison[comparison["profile_total_trips"].notna()]
def relative_level(frame):
    return frame["total_trips"].sum() / frame["profile_total_trips"].sum()
profile = (
    comparison.groupby(["group", "g_closure_hour", "hour_of_day"]).apply(relative_level, include_groups=False)
    .rename("relative_trips").reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
styles = [(False, "Weekend hours with G service", "tab:blue"), (True, "Weekend hours with G reduction (MTA notice)", "tab:red")]
for axis, group in zip(axes, ["Exposed (within 0.25 mile)", "Other corridor", "Control (more than 3 miles)"]):
    for closure_flag, label, color in styles:
        subset = profile[(profile["group"] == group) & (profile["g_closure_hour"] == closure_flag)]
        axis.plot(subset["hour_of_day"], subset["relative_trips"], marker="o", markersize=3, color=color, label=label)
    axis.axhline(1, color="black", linewidth=0.8, linestyle=":")
    axis.set_title(group)
    axis.set_xlabel("Hour of day")
    axis.grid(alpha=0.3)
axes[0].set_ylabel("Actual trips relative to the station normal profile")
axes[0].legend(fontsize=8)
fig.suptitle("Weekend trips with and without a confirmed G reduction, dry hours, 13 weekends in 2026")
fig.tight_layout()
plt.show()

closure_summary = (
    comparison.groupby(["group", "g_closure_hour"]).apply(
        lambda frame: pd.Series({"actual to profile": relative_level(frame), "station hours": len(frame)}), include_groups=False)
    .unstack("g_closure_hour")
)
closure_summary.columns = [f"{stat} ({'closure' if flag else 'service running'})" for stat, flag in closure_summary.columns]
closure_summary["closure to running ratio"] = closure_summary["actual to profile (closure)"] / closure_summary["actual to profile (service running)"]
display(closure_summary.round(3))
ratio_control = closure_summary.loc["Control (more than 3 miles)", "closure to running ratio"]
for group in ("Exposed (within 0.25 mile)", "Other corridor"):
    print(f"Ratio of ratios, {group} relative to control: {closure_summary.loc[group, 'closure to running ratio'] / ratio_control:.3f}")
print(f"Closure weekends in the comparison: {comparison.loc[comparison['g_closure_hour'], 'hour'].dt.normalize().nunique()} days")




**Interpretation of Figure 6.** The figure shows actual trips relative to each station's normal profile, by hour of day, for weekend hours with and without a confirmed G reduction. Values above 1 mean more trips than the profile. Control stations more than 3 miles from the corridor move from 1.100 to 1.182 (ratio 1.074), which captures the general lift of closure weekends, for example from season and weather. Exposed stations within 0.25 mile move from 1.116 to 1.391 (ratio 1.247), and other corridor stations from 1.113 to 1.336 (ratio 1.200). Relative to control, the ratio of ratios is 1.161 for exposed stations and 1.117 for other corridor stations, so demand rose about 16 percent more near the closed segment than at control stations and the lift fades with distance. The comparison covers 24 closure days and uses dry hours only, so weather is held roughly constant. This agrees with the screening results in Section 2.6 (1.17 for 2026 weekends). It is a descriptive difference, not a causal estimate, because events and other weekend effects are not controlled. It motivates a count model with station, hour and weather controls.

## 3.7 Weather and the distribution of the outcome

Figure 7 relates weather to demand. Because demand depends strongly on the time of day, departures are expressed relative to the average for the same station, hour of day and day type before they are grouped by weather class. Figure 8 examines the distribution of hourly departures, because the choice of count model in the next stage depends on whether the variance is close to the mean.


In [ ]:
active = panel[panel["trip_observed"] & panel["hour_of_day"].between(6, 23)].copy()
expected_demand = active.groupby(["station_id", "hour_of_day", "is_weekend"])["departures"].transform("mean")
active["relative_departures"] = active["departures"] / expected_demand.replace(0, np.nan)
active["precipitation_class"] = np.select(
    [active["precipitation"].eq(0), active["precipitation"].le(1)],
    ["Dry", "Light rain (up to 1 mm)"],
    default="Moderate or heavy (over 1 mm)",
)
active["temperature_quartile"] = pd.qcut(active["temperature_2m"], 4)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for axis, column, order in (
    (axes[0], "precipitation_class", ["Dry", "Light rain (up to 1 mm)", "Moderate or heavy (over 1 mm)"]),
    (axes[1], "temperature_quartile", None),
):
    summary = active.groupby(column, observed=True)["relative_departures"].agg(["mean", "count"])
    if order:
        summary = summary.reindex(order)
    axis.bar(range(len(summary)), summary["mean"], color="tab:blue")
    axis.set_xticks(range(len(summary)))
    axis.set_xticklabels([f"{str(label)}\n(n = {int(n):,})" for label, n in zip(summary.index, summary["count"])], fontsize=8)
    axis.axhline(1, color="black", linewidth=0.8, linestyle="--")
    axis.set_xlabel("Precipitation class" if column == "precipitation_class" else "Temperature quartile (degrees Celsius)")
axes[0].set_ylabel("Departures relative to the station, hour and day type average")
fig.suptitle("September relative departures by weather condition, 06:00 to 23:59")
fig.tight_layout()
plt.show()

weather_summary = active.groupby("precipitation_class")["relative_departures"].agg(["mean", "count"]).round(3)
display(weather_summary)
print(f"Hours with any precipitation (all panel hours): {int((panel.drop_duplicates("hour")["precipitation"] > 0).sum())}")


**Interpretation of Figure 7.** Departures are scaled by the station, hour and day type average, so a value of 1 means typical demand. Dry hours average 1.08, light rain hours 0.83 and moderate or heavy rain hours 0.47. Demand falls steadily as precipitation rises, and the moderate or heavy class has far fewer observations (2,779) than the dry class (35,449). The rain effect is larger than the closure difference in Figure 6, so precipitation has to be kept as a control in any model, otherwise a rainy closure weekend could be mistaken for a closure effect. The weather grid is coarse relative to station locations, and the pattern is descriptive.

In [ ]:
observed = panel[panel["trip_observed"]]
departures_mean = observed["departures"].mean()
departures_var = observed["departures"].var()
zero_share = observed["departures"].eq(0).mean()
station_moments = observed.groupby("station_id")["departures"].agg(["mean", "var"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
upper = int(observed["departures"].quantile(0.995))
axes[0].hist(observed["departures"].clip(upper=upper), bins=np.arange(0, upper + 2) - 0.5, color="tab:blue")
axes[0].set_yscale("log")
axes[0].set_xlabel(f"Departures per station hour (values above {upper} grouped at {upper})")
axes[0].set_ylabel("Station hours (log scale)")
axes[0].set_title("Distribution of hourly departures")
limit = float(station_moments[["mean", "var"]].max().max()) * 1.05
axes[1].scatter(station_moments["mean"], station_moments["var"], color="tab:blue")
axes[1].plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=0.8, label="Variance equals mean (Poisson)")
axes[1].set_xlabel("Station mean of hourly departures")
axes[1].set_ylabel("Station variance of hourly departures")
axes[1].set_title("Mean and variance by station")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"Mean departures per station hour: {departures_mean:.3f}")
print(f"Variance of departures per station hour: {departures_var:.3f}")
print(f"Variance to mean ratio: {departures_var / departures_mean:.2f}")
print(f"Share of station hours with zero departures: {zero_share:.1%}")
print(f"Median station level variance to mean ratio: {(station_moments['var'] / station_moments['mean']).median():.2f}")


**Interpretation of Figure 8.** The left panel shows the distribution of departures per station hour and the right panel the relation between the mean and the variance by station. Station hours average 4.99 departures with a variance of 38.08, a variance to mean ratio of 7.63, and 28.0 percent of station hours have zero departures. The median station level ratio is 4.96. A Poisson count has a ratio of 1, so the outcome is strongly overdispersed with excess zeros. This points toward a negative binomial or zero inflated count model, or a Poisson model with robust errors, rather than ordinary least squares on raw counts. Part of the overdispersion comes from the daily cycle and differences between stations, which later features will absorb, so the ratio is an upper bound on the dispersion that remains after modeling.

## 3.8 Dataset readiness for modeling

This section is the hand off to the modeling sections. It lists every ready table, says which one to start from, and gives the rules to follow before fitting anything. All ready tables are Delta tables in the project schema (default `workspace.citibike`), built from the raw files by the `citibike-prepare` job, which runs after each weekly download and can be started on its own. The raw downloads stay in the volume, and each raw reference file also has a `raw_` table.

### Where to start

For almost all modeling work, load one table: the extended panel. The other tables are lookups or summaries that explain how it was built.

```python
from citibike import data_access

panel = data_access.read_pandas("analysis_station_hour_panel_extended")
panel = panel[panel["station_active"]]                      # drop hours before a station opened or after it closed
train = panel[panel["split"] == "train"]                    # January to June 2026
valid = panel[panel["split"] == "validation"]               # July 2026
test = panel[panel["split"] == "test"]                      # August and September 2026
check_2024 = panel[panel["split"] == "validation_2024"]     # May to September 2024, external check
```

### Catalog of ready tables

| Table | Rows | Grain | Use it for |
|---|---|---|---|
| `analysis_station_hour_panel_extended` | 1,485,072 | station by clock hour, 153 stations | Main modeling table: outcomes, weather, calendar, shutdown flags, baseline, split. |
| `trip_hourly_model_stations` | 917,942 | station (`short_name`) by hour, only hours with trips | Intermediate hourly counts from the 58 million trip rows. Rebuild the panel from it; not needed for modeling. |
| `analysis_station_hour_panel` | 62,952 | station by hour, 43 Greenpoint stations, August and September 2026 | The Section 2.5 panel, the only one with Citi Bike status (bikes and docks available). |
| `mta_subway_shutdowns` | 118 | one event by one affected G stop | Shutdown table from the proposal: event id, type, stop, start and end time, nearby stations. |
| `model_station_sets` | 154 | station | Station lookup with `station_group` (94 corridor, 60 control) and distance to the nearest G stop. |
| `greenpoint_station_exposure` | 43 | Greenpoint station | Distance to the nearest affected stop, `exposed_station` and `distance_band`. |
| `g_service_closures_confirmed` | 4 | closure weekend | The August and September 2026 closure windows from MTA notices. |
| `g_service_closures_confirmed_pre_august` | 9 | closure weekend | The nine earlier 2026 closure weekends (`times_assumed` marks inferred times). |
| `g_service_reduction_windows_inferred` | 8 | window | Windows inferred from ridership, kept only as a cross check on the notices. |
| `baseline_accuracy_by_month` | 24 | month by station group | Error of the four week baseline, the reference every model should beat. |
| `shutdown_effect_screening` | 10 | event by group | Actual to profile demand ratio for each shutdown event, the descriptive effect behind Figure 6. |

`model_station_sets` has 154 rows because it includes station `JC124`, which has no trips and so is absent from the panels (153 stations).

### Keys and how to join

- `station_id` (GBFS identifier) and `hour` (naive New York local time) identify one row of every panel. The pair is unique.
- Trip files use the short station code `short_name` (for example `4665.02`). Status files and the Section 2.5 panel use `station_id`. `model_station_sets` maps one to the other. The extended panel carries both.
- `event_id` links the panel to `mta_subway_shutdowns`. There are 16 events: three 2024 phases (E01 to E03) and thirteen 2026 closure weekends.

### Extended panel columns

| Group | Columns | Meaning |
|---|---|---|
| Keys and design | `station_id`, `short_name`, `hour`, `period`, `split`, `station_group` | `period` is 2024 or 2026. `split` is chronological: `train` 655,944 rows, `validation` 112,344, `test` 221,064, `validation_2024` 495,720. `station_group` is `corridor` (within reach of the affected G segment, 94 stations) or `control` (59 stations with trips). |
| Outcomes | `departures`, `arrivals`, `total_trips`, `net_flow` | Hourly counts. `net_flow` is arrivals minus departures. Zero is filled for hours without trips. |
| Weather | `temperature_2m`, `precipitation`, `precipitation_any` | Hourly values from one Open Meteo grid cell, the same for all stations. |
| Calendar | `hour_of_day`, `hour_sin`, `hour_cos`, `day_of_week`, `is_weekend`, `is_holiday` | The sine and cosine pair encodes the daily cycle. |
| Station attributes | `name`, `lat`, `lon`, `capacity`, `miles_to_nearest_g_stop` | From the latest station snapshot. |
| Shutdown flags | `g_closure_hour`, `exposed_closure_hour`, `event_id`, `event_type`, `event_exposed`, `distance_to_event_miles`, `exposed_station`, `exposed_station_2024`, `placebo_weekend_hour` | Whether the hour falls in a closure, whether the station is within a quarter mile of the affected stops, and the matching weekend hours with no closure used as placebo. |
| Reference demand | `baseline_departures`, `baseline_arrivals`, `baseline_total_trips`, `profile_departures`, `profile_arrivals`, `profile_total_trips` | `baseline_*` is the mean of the same hour in the previous four weeks, closure hours excluded, and needs at least two of them. `profile_*` is the mean per station, day of week and hour over a normal reference window. Both are expectations of normal demand, not outcomes. |
| Status | `bikes_available_mean`, `docks_available_mean`, `zero_bike_snapshots`, `zero_dock_snapshots`, `snapshot_count`, `status_observed`, `bikes_change`, `status_net_proxy`, `status_departure_proxy`, `status_arrival_proxy` | Filled only for the 43 Greenpoint stations after August 21, 2026 (about 3 percent of rows). `bikes_change` is the change in bikes available and the `status_*_proxy` columns are the demand proxies built from it. |
| Quality | `station_active` | False for 72,192 rows (4.9 percent) before a station's first trip or after its last. These zeros are not demand. |

The Section 2.5 panel keeps its own names for the same ideas (`zero_bike_share`, `zero_dock_share`, `departures_lag_24h`, `departures_lag_168h`, `nearest_affected_stop`, `nearest_affected_stop_miles`, `distance_band`, `trip_observed`). Use it when status data is the focus.

### Rules before modeling

1. Filter `station_active` to True before any fit or score. Figures 6, 9 and 10 and the baseline already do.
2. Never use `baseline_*`, `profile_*` or `status_*` columns as predictors of the same hour's outcome without checking leakage. The baseline uses only earlier weeks, but the profile uses the reference window and the status proxies are derived from the outcome.
3. Fit on `train`, tune on `validation`, report on `test`. Use `validation_2024` only to check that a shutdown effect also appears in a different year.
4. The baseline is missing for 253,793 of the 495,720 station hours in 2024 (the first four weeks of May, and hours whose earlier weeks fall in a closure or before a station opened). Check for missing values, or use the profile.
5. The outcome is a strongly overdispersed count (Figure 8), so start with a Poisson or negative binomial model and compare with the baseline errors in `baseline_accuracy_by_month`.
6. Treat the shutdown effect results as descriptive. Corridor and control stations differ in demand level, so compare ratios to the profile, as in Figure 6, and not raw counts.

### Known limitations

- 13 of the 16 closure weekends proposed were found in MTA notices. Eight of the earlier weekends use assumed times, and the 2024 phase dates come from web notices and use whole days.
- Station status covers only Greenpoint from August 21 to September 30, 2026, and three of the four August and September closure weekends overlap it.
- One weather grid cell serves all stations, and weather on closure weekends was not matched to comparison weekends.
- Station attributes come from the latest snapshot. Exposure uses distance to a stop, not ridership by station.
- Skipped stop work on the southern F and G section is not treated as a closure.
- Trip loading reads four fields only, so there is no outlier removal and no duplicate ride id check. Daylight saving transitions are assumed ignorable.

## 3.9 Exploration summary

The local files support a descriptive baseline for 43 stations in the Greenpoint screening rectangle. September activity was higher than January, and the hourly profiles show different weekday and weekend peaks. Zero open docks were reported more often than zero bikes.

Four MTA Weekender notices (August 7 to 10, August 21 to 24, September 11 to 14 and September 25 to 28) confirm a G suspension between Court Sq and Bedford-Nostrand Avs. Hourly ridership agrees with the notices: all inferred reduction hours fall inside the confirmed windows. Eight G stops are affected, and 14 of 43 stations lie within a quarter mile of one. The analysis panel (62,952 station hour rows from August 1 to September 30) joins trips, status, weather, exposure and closure flags. Figure 6 uses the extended panel, the 13 confirmed 2026 weekends and dry hours, and gives a ratio of ratios of 1.16 for exposed stations and 1.12 for other corridor stations relative to control. Rain lowers relative demand strongly (0.47 for moderate or heavy rain against 1.08 dry). Departures are overdispersed (variance to mean ratio 7.6, 28 percent zeros).

Limits: 13 of the 16 closure weekends were found (eight use assumed times), status starts on August 21, one weather cell serves all stations, and weather on closure weekends was not matched to comparison weekends. These results are descriptive and do not establish a causal effect. Section 2.6 extends the data to 153 stations, 14 trip months and 16 shutdown events, adds a four week baseline and a normal profile, and gives a screening result: demand at exposed stations relative to control stations is 1.01 on placebo weekends, 1.17 on confirmed 2026 closure weekends and 1.12 in 2024 phase 2. Next steps are to fit count models with station, hour and weather controls on the chronological split.


# 4. Model development


## 4.1 Modeling design

The model is a counterfactual demand forecaster. It learns normal hourly demand at each station from hours when the G train was running. During a closure it predicts what demand would have been with normal service. The ratio of actual to predicted demand at exposed stations, compared with control stations, measures how demand shifts during a disruption.

Design choices:

1. **Unit and target.** One station in one hour. The first target is `departures`; `arrivals` follows the same pipeline.
2. **Training data.** The `train` split (January to June 2026), active stations only, with every closure hour removed. A counterfactual model must never see disrupted hours, or it learns the effect it is supposed to measure.
3. **Tuning and testing.** Tune on `validation` (July 2026). Report accuracy on normal hours in `test` (August and September 2026). Closure hours in `test` are scored separately to estimate the disruption effect.
4. **Reference.** Every model must beat the four week baseline (`baseline_departures`).
5. **Predictors.** Only information known at forecast time: calendar, weather, station attributes and station identity. The `baseline_*`, `profile_*` and `status_*` columns are not used as predictors.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

panel = read_table("analysis_station_hour_panel_extended").sort_values(["short_name", "period", "hour"]).reset_index(drop=True)

# Rows before a station opened or after it closed are zeros, not demand
panel = panel[panel["station_active"]].copy()

# Closure hours: any hour inside a confirmed G closure window
closure = panel["g_closure_hour"].fillna(False).astype(bool)
panel["is_closure"] = closure

splits = ["train", "validation", "test", "validation_2024"]
summary = (
    panel.groupby(["split", "is_closure"])
    .size()
    .unstack(fill_value=0)
    .reindex(splits)
)
summary.columns = ["normal_hours", "closure_hours"]
print(summary)
print("\nStations:", panel["station_id"].nunique())
print("Station groups:\n", panel.drop_duplicates("station_id")["station_group"].value_counts())

## 4.2 Clean training data and the baseline to beat

Closure hours are removed from every split used to fit or tune the model. They are kept aside in `test_closure` to estimate the disruption effect later. The four week baseline (`baseline_departures`) is scored on the same normal hours as the models, so the comparison is fair. The baseline is missing when fewer than two of the previous four weeks are usable, so its coverage is reported with the error.

In [ ]:
TARGET = "departures"

train_df = panel[(panel["split"] == "train") & ~panel["is_closure"]].copy()
valid_df = panel[(panel["split"] == "validation") & ~panel["is_closure"]].copy()
test_df = panel[(panel["split"] == "test") & ~panel["is_closure"]].copy()
test_closure = panel[(panel["split"] == "test") & panel["is_closure"]].copy()

print(f"train normal:  {len(train_df):,}")
print(f"valid normal:  {len(valid_df):,}")
print(f"test normal:   {len(test_df):,}")
print(f"test closure:  {len(test_closure):,}")

def score(y_true, y_pred):
    err = y_true - y_pred
    return {
        "MAE": np.abs(err).mean(),
        "RMSE": np.sqrt((err ** 2).mean()),
        "mean_actual": y_true.mean(),
        "mean_pred": y_pred.mean(),
    }

def baseline_score(df, name):
    has_base = df["baseline_departures"].notna()
    result = score(df.loc[has_base, TARGET], df.loc[has_base, "baseline_departures"])
    result["coverage"] = has_base.mean()
    return pd.Series(result, name=name)

baseline_results = pd.concat(
    [baseline_score(valid_df, "validation"), baseline_score(test_df, "test")], axis=1
).T
print("\nFour week baseline on normal hours")
print(baseline_results.round(3))

print("\nTarget summary in train")
print(train_df[TARGET].describe().round(2))

## 4.3 Poisson regression models

Departures are counts with many zeros and a long right tail (Figure 8), so the first models are Poisson regressions with a log link. Model A uses station identity, hour of day by day type, day of week, holidays and weather. Model B adds the logged four week baseline, which carries recent demand level and absorbs the seasonal rise from the January to June training months into July. The baseline is computed from earlier weeks with closure hours excluded, so it is available at forecast time and stays a valid input during a closure.

All models are fit on normal training hours with a baseline and scored on normal validation hours with a baseline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor

def add_features(df):
    df = df.copy()
    df["hour_daytype"] = df["hour_of_day"].astype(str) + "_" + df["is_weekend"].astype(int).astype(str)
    df["temp_sq"] = df["temperature_2m"] ** 2
    df["is_holiday"] = df["is_holiday"].astype(int)
    df["precipitation_any"] = df["precipitation_any"].astype(int)
    df["log_baseline"] = np.log1p(df["baseline_departures"])
    return df

# Keep rows where the baseline exists, so every model is scored on the same hours
fit_df = add_features(train_df[train_df["baseline_departures"].notna()])
val_df = add_features(valid_df[valid_df["baseline_departures"].notna()])

CAT = ["station_id", "hour_daytype", "day_of_week"]
NUM_A = ["temperature_2m", "temp_sq", "precipitation", "precipitation_any", "is_holiday"]
NUM_B = NUM_A + ["log_baseline"]

def make_model(num_cols):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ("num", StandardScaler(), num_cols),
    ])
    return Pipeline([("prep", prep), ("glm", PoissonRegressor(alpha=1e-4, max_iter=500))])

model_a = make_model(NUM_A).fit(fit_df[CAT + NUM_A], fit_df[TARGET])
model_b = make_model(NUM_B).fit(fit_df[CAT + NUM_B], fit_df[TARGET])

val_df["pred_a"] = model_a.predict(val_df[CAT + NUM_A])
val_df["pred_b"] = model_b.predict(val_df[CAT + NUM_B])

results_val = pd.DataFrame({
    "baseline": score(val_df[TARGET], val_df["baseline_departures"]),
    "poisson_A": score(val_df[TARGET], val_df["pred_a"]),
    "poisson_B": score(val_df[TARGET], val_df["pred_b"]),
}).T
print(f"Training rows: {len(fit_df):,}   Validation rows: {len(val_df):,}")
print(results_val.round(3))

## 4.4 Gradient boosted Poisson model

Model B matched the baseline on RMSE but overpredicted July by about 10 percent, and Model A overpredicted more. A linear model in log space extrapolates temperature effects beyond the January to June training range. A gradient boosted tree model with a Poisson objective avoids this: tree predictions stay flat outside the observed range, and station by hour interactions are learned directly. The validation set is used for early stopping only.

In [ ]:
import lightgbm as lgb

GBM_FEATURES = [
    "station_id", "hour_of_day", "is_weekend", "day_of_week", "is_holiday",
    "temperature_2m", "precipitation", "precipitation_any",
    "capacity", "lat", "lon", "log_baseline",
]
station_categories = pd.CategoricalDtype(sorted(panel["station_id"].unique()))

def gbm_frame(df):
    X = df[GBM_FEATURES].copy()
    X["station_id"] = X["station_id"].astype(station_categories)
    X["day_of_week"] = X["day_of_week"].astype(str).astype("category")
    return X

X_fit, X_val = gbm_frame(fit_df), gbm_frame(val_df)
X_val["day_of_week"] = X_val["day_of_week"].cat.set_categories(X_fit["day_of_week"].cat.categories)

gbm = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    verbose=-1,
)
gbm.fit(
    X_fit, fit_df[TARGET],
    eval_set=[(X_val, val_df[TARGET])],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
val_df["pred_gbm"] = gbm.predict(X_val, num_iteration=gbm.best_iteration_)

results_val.loc["lightgbm"] = score(val_df[TARGET], val_df["pred_gbm"])
print(f"Best iteration: {gbm.best_iteration_}")
print(results_val.round(3))

## 4.5 Counterfactual demand during G closures

The LightGBM model predicts what departures would have been with normal G service. For each station group, the demand ratio is total actual departures divided by total predicted departures. Stations are grouped as exposed (within a quarter mile of an affected G stop), other corridor, and control. Model bias common to all stations cancels in the ratio relative to control.

The same calculation on test weekends with no closure (placebo weekends) should give a relative ratio near 1. A closure ratio clearly above the placebo ratio indicates a shift in demand during the disruption.

In [ ]:
def station_group_label(df):
    exposed = df["exposed_station"].fillna(False).astype(bool)
    return np.where(exposed, "exposed", np.where(df["station_group"] == "corridor", "other corridor", "control"))

def counterfactual_ratios(df, label):
    df = add_features(df[df["baseline_departures"].notna()])
    X = gbm_frame(df)
    X["day_of_week"] = X["day_of_week"].cat.set_categories(X_fit["day_of_week"].cat.categories)
    df["pred_gbm"] = gbm.predict(X, num_iteration=gbm.best_iteration_)
    df["group"] = station_group_label(df)
    out = df.groupby("group").agg(actual=(TARGET, "sum"), predicted=("pred_gbm", "sum"), station_hours=(TARGET, "size"))
    out["ratio"] = out["actual"] / out["predicted"]
    out["relative_to_control"] = out["ratio"] / out.loc["control", "ratio"]
    out.insert(0, "window", label)
    return out.reindex(["exposed", "other corridor", "control"]), df

placebo_df = test_df[test_df["placebo_weekend_hour"].fillna(False).astype(bool)]

closure_table, closure_scored = counterfactual_ratios(test_closure, "closure weekends")
placebo_table, placebo_scored = counterfactual_ratios(placebo_df, "placebo weekends")

effect_table = pd.concat([closure_table, placebo_table])
print(effect_table.round(3))

The placebo check fails for the control group. Control stations lie 3 to 9 miles from the G (the Bronx, upper Manhattan, Bay Ridge, Queens and Jersey City), carry under one trip per hour, and are overpredicted by 25 to 35 percent even on normal weekends. Dividing by their ratio inflates every relative value, so the placebo gives 1.34 instead of 1.

Exposed and other corridor stations are fit well on placebo weekends (ratios 1.017 and 1.013), so other corridor stations are used as the comparison group. Closure ratio divided by placebo ratio is 1.019 for exposed stations and 0.993 for other corridor stations, which gives a relative shift of about 1.026. Departures near closed G stops run roughly 2 to 3 percent above the normal service counterfactual, relative to nearby corridor stations. Corridor stations may also respond to the closure, so this is likely a lower bound. The control group should be rebuilt from stations matched on distance and demand level.

# 5. Evaluation and validation

## 5.1 Effect by closure weekend

A pooled ratio can hide disagreement between events. The exposed to corridor ratio is computed separately for each of the four test closure weekends to show whether the shift is consistent.

In [ ]:
closure_scored["weekend"] = closure_scored["event_id"]
per_weekend = (
    closure_scored[closure_scored["group"] != "control"]
    .groupby(["weekend", "group"])
    .agg(actual=(TARGET, "sum"), predicted=("pred_gbm", "sum"))
)
per_weekend["ratio"] = per_weekend["actual"] / per_weekend["predicted"]
per_weekend = per_weekend["ratio"].unstack("group")
per_weekend["exposed_vs_corridor"] = per_weekend["exposed"] / per_weekend["other corridor"]
print(per_weekend.round(3))

## 5.2 Placebo distribution and the size of normal variation

Each normal weekend in validation and test gives one placebo value of the exposed to corridor ratio. Their spread shows how much the ratio varies without a closure. A closure effect is credible only if the closure weekends fall outside most of that spread.

In [ ]:
placebo_all = pd.concat([
    valid_df[valid_df["placebo_weekend_hour"].fillna(False).astype(bool)],
    test_df[test_df["placebo_weekend_hour"].fillna(False).astype(bool)],
])
_, placebo_all_scored = counterfactual_ratios(placebo_all, "placebo")

# Shift by one day so Monday morning hours stay with their weekend
placebo_all_scored["weekend"] = (
    placebo_all_scored["hour"] - pd.Timedelta(days=1)
).dt.to_period("W").astype(str)

placebo_weekends = (
    placebo_all_scored[placebo_all_scored["group"] != "control"]
    .groupby(["weekend", "group"])
    .agg(actual=(TARGET, "sum"), predicted=("pred_gbm", "sum"))
)
placebo_weekends["ratio"] = placebo_weekends["actual"] / placebo_weekends["predicted"]
placebo_weekends = placebo_weekends["ratio"].unstack("group")
placebo_weekends["exposed_vs_corridor"] = placebo_weekends["exposed"] / placebo_weekends["other corridor"]

print(placebo_weekends.round(3))
print("\nPlacebo exposed_vs_corridor: mean {:.3f}, sd {:.3f}, n {}".format(
    placebo_weekends["exposed_vs_corridor"].mean(),
    placebo_weekends["exposed_vs_corridor"].std(),
    len(placebo_weekends),
))
print("Closure exposed_vs_corridor: mean {:.3f}, n {}".format(
    per_weekend["exposed_vs_corridor"].mean(), len(per_weekend)
))

## 5.3 Is the closure shift larger than normal variation?

Two tests compare the four closure weekends with the eight placebo weekends. Welch's t test does not assume equal variance. The exact permutation test makes no distributional assumption: it relabels every possible set of four weekends out of twelve as closures (495 sets) and counts how often the difference in means is at least as large as the observed one. Both tests are one sided, because the hypothesis is that demand near closed stops rises.

In [ ]:
from itertools import combinations
from scipy import stats

closure_vals = per_weekend["exposed_vs_corridor"].to_numpy()
placebo_vals = placebo_weekends["exposed_vs_corridor"].to_numpy()
observed_diff = closure_vals.mean() - placebo_vals.mean()

welch = stats.ttest_ind(closure_vals, placebo_vals, equal_var=False, alternative="greater")

pooled = np.concatenate([closure_vals, placebo_vals])
n_closure = len(closure_vals)
diffs = []
for idx in combinations(range(len(pooled)), n_closure):
    mask = np.zeros(len(pooled), dtype=bool)
    mask[list(idx)] = True
    diffs.append(pooled[mask].mean() - pooled[~mask].mean())
diffs = np.array(diffs)
perm_p = (diffs >= observed_diff - 1e-12).mean()

print(f"Closure mean {closure_vals.mean():.3f}   Placebo mean {placebo_vals.mean():.3f}")
print(f"Observed shift: {observed_diff * 100:.1f} percent")
print(f"Welch t = {welch.statistic:.2f}, one sided p = {welch.pvalue:.3f}")
print(f"Exact permutation p = {perm_p:.3f} ({len(diffs)} relabelings)")

## 5.4 Accuracy on held out normal hours

The test split (August and September 2026) was not used for fitting or tuning. Models are scored on its normal hours with a baseline, overall and by station group, to check that the counterfactual is accurate where the effect is measured.

In [ ]:
test_scored = add_features(test_df[test_df["baseline_departures"].notna()])
X_test = gbm_frame(test_scored)
X_test["day_of_week"] = X_test["day_of_week"].cat.set_categories(X_fit["day_of_week"].cat.categories)
test_scored["pred_gbm"] = gbm.predict(X_test, num_iteration=gbm.best_iteration_)
test_scored["pred_b"] = model_b.predict(test_scored[CAT + NUM_B])
test_scored["group"] = station_group_label(test_scored)

results_test = pd.DataFrame({
    "baseline": score(test_scored[TARGET], test_scored["baseline_departures"]),
    "poisson_B": score(test_scored[TARGET], test_scored["pred_b"]),
    "lightgbm": score(test_scored[TARGET], test_scored["pred_gbm"]),
}).T
print(f"Test normal hours: {len(test_scored):,}")
print(results_test.round(3))

by_group = test_scored.groupby("group").apply(
    lambda g: pd.Series({
        "baseline_RMSE": score(g[TARGET], g["baseline_departures"])["RMSE"],
        "lightgbm_RMSE": score(g[TARGET], g["pred_gbm"])["RMSE"],
        "actual_over_pred": g[TARGET].sum() / g["pred_gbm"].sum(),
    }),
    include_groups=False,
).reindex(["exposed", "other corridor", "control"])
print("\nBy station group")
print(by_group.round(3))

## 5.5 Evaluation summary and limitations

**Forecast accuracy.** On held out normal hours in August and September 2026, the LightGBM Poisson model has RMSE 2.667 against 2.863 for the four week baseline, an improvement of about 7 percent, with nearly no bias (mean prediction 3.258 against 3.255 actual). It beats the baseline in every station group. Poisson regression with the baseline as a feature also beats the baseline on test RMSE (2.703) but overpredicts by about 4 percent.

**Validity of the comparison.** On normal test hours the actual to predicted ratio is 1.033 for exposed stations and 1.034 for other corridor stations. The model errs the same way for both groups, so the exposed to corridor ratio isolates what differs between them during a closure. Control stations are overpredicted by about 21 percent (ratio 0.788) and are not used as a comparison.

**Disruption effect.** Across the four test closure weekends, departures at exposed stations relative to corridor stations average 1.031, against 1.006 on eight placebo weekends: a shift of about 2.5 percent. All four closure weekends are above 1, and one (E15, 1.062) exceeds every placebo weekend. The shift is about the size of normal weekend variation (placebo sd 0.025). Welch's t test gives one sided p = 0.068, and the exact permutation test gives p = 0.065. The evidence is suggestive but not significant at the 5 percent level.

**Limitations.**

1. Only four closure weekends are out of sample. Each new closure in the production period adds evidence.
2. Corridor stations may also respond to closures (E14 and E15 run 10 to 15 percent above prediction in both groups), so the relative shift is likely a lower bound on the local effect.
3. The control group (3 to 9 miles from the G, including Jersey City) is poorly fit and should be rebuilt from stations matched on distance and demand level.
4. One weather grid cell serves all stations, and the model missed weekend E16 by about half in every group, probably because of rain.
5. Eight of the earlier 2026 closure weekends use assumed times, which affects which training hours were removed.
6. Only departures are modeled. Arrivals and station availability (empty and full docks) are the next targets.

# 6. Production design and monitoring



# 7. Results and discussion



# 8. Conclusion



# 9. References and appendices

1. Citi Bike. Citi Bike System Data. https://citibikenyc.com/system-data
2. Citi Bike. Trip history archives for January, August and September 2026. https://s3.amazonaws.com/tripdata/202601-citibike-tripdata.zip, https://s3.amazonaws.com/tripdata/202608-citibike-tripdata.zip and https://s3.amazonaws.com/tripdata/202609-citibike-tripdata.zip
3. New York State Open Data. MTA Service Alerts: Beginning April 2020. https://data.ny.gov/resource/7kct-peq7.json. The API query retrieves New York City Transit Subway alert updates dated January 1 through October 1, 2026; the available records end August 11, 2026.
4. CUNY SPS Data Science. Citi Bike data pipeline. https://github.com/cuny-sps-datascience/citibike-data-pipeline. Reference supplied for project data collection and processing.

5. Google Cloud. BigQuery public datasets `bigquery-public-data.new_york_citibike.citibike_stations` (https://console.cloud.google.com/bigquery?project=eliot-463900&ws=!1m6!1m5!4m3!1sbigquery-public-data!2snew_york_citibike!3scitibike_stations!23sRESOURCE_LIST) and `bigquery-public-data.new_york_citibike.citibike_trips` (https://console.cloud.google.com/bigquery?project=eliot-463900&ws=!1m6!1m5!4m3!1sbigquery-public-data!2snew_york_citibike!3scitibike_trips!23sRESOURCE_LIST). The local station status data under `datasource/station_status` is an NYC export partitioned by `nyc_date`; the file metadata (status poller version 1.2, Spark export job written on October 1, 2026) is recorded in the data sources document (`docs/data_sources.md`).
6. Open Meteo. Historical Weather API. https://open-meteo.com/en/docs/historical-weather-api. Hourly temperature and precipitation are requested for the Greenpoint station coordinates in January, August and September 2026. The returned grid coordinates are saved in a station weather lookup.
7. New York State Open Data. MTA Subway Stations. https://data.ny.gov/resource/39hk-dx4f.json. The local export `MTA_Subway_Stations_20261003.csv` provides stop identifiers, route labels, names, and coordinates; stops with G service are selected from it.
8. MTA. MTA Weekender notices for August 7 to 10, August 21 to 24, September 11 to 14 and September 25 to 28, 2026. https://www.mta.info/article/mta-weekender-august-7-10-2026, https://www.mta.info/article/mta-weekender-august-21-24-2026, https://www.mta.info/article/mta-weekender-september-11-14-2026 and https://www.mta.info/article/mta-weekender-september-25-28-2026
9. MTA. G line summer 2024 service change, June 28 to September 3, 2024, in three phases; and MTA Weekender notices for 2026 weekends with no G trains between Court Sq and Bedford-Nostrand Avs (February 27, March 6, 13, 20 and 27, April 17, June 5, June 19 and July 10), using the same URL pattern as reference 8. Citi Bike trip archives for May to September 2024 and February to July 2026 follow the pattern in reference 2.

Appendices can hold further data definitions, cleaning checks, and supplementary results.